# Vision Model → Segmentation Taxonomy — v1.5.7.1 SHARE-READY AUDITED

**Share-ready audited build derived from v1.5.7.1 SHARE-READY AUDITED.**

## What is frozen in this build

- v0.9 semantic-backbone architecture
- Google Street View artifact scientific `IGNORE=255`
- border-fringe Google residual correction
- tree/glazing protection
- woody trunk/branch recovery
- scaffold precedence guard
- awning / wall / fence recovery
- false-sidewalk → roadway local cleanup
- bike-lane pixels merged into roadway in final outputs
- non-overlapping final summary layout

## Share-ready consistency rule

After every final semantic cleanup, the authoritative scientific state is
re-committed from `final_working` into `final_v010` before QA/statistics/export.

Therefore the following now use the SAME final classification state:

1. scientific label PNG
2. pseudo-label PNG
3. statistics CSV
4. diagnostics JSON
5. QA percentages
6. Combined Semantic Mask / final summary

## Runtime

Designed for **Google Colab + GPU**.  
Input mode in this notebook is **single-image inference**.


In [ ]:
!pip -q install "transformers>=4.57,<5" "huggingface-hub<1.0" accelerate safetensors pandas matplotlib pillow scikit-learn tqdm opencv-python torchvision
print("✓ Dependencies installed")


In [ ]:
!pip uninstall -y gradio gradio-client -q

print("✓ Unused Gradio packages removed")


In [ ]:
from pathlib import Path
import json, gc, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path("/content/VLM_Street_Interface_Segmentation")
CONFIG_DIR = PROJECT_ROOT / "config"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for d in [PROJECT_ROOT, CONFIG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_BASELINE_VERSION = "hybrid_v0.9_semantic_backbone"
SHARE_VERSION = "v1.5.7.1-share-ready-audited"
TAXONOMY_VERSION = "street_interface_v1.5.7.1"
IGNORE_INDEX = 255

CLASS_NAMES = {0: 'other_unknown',
 1: 'roadway',
 2: 'sidewalk',
 3: 'bike_lane',
 4: 'curb_edge',
 5: 'upper_building_facade',
 6: 'ground_floor_solid_facade',
 7: 'ground_floor_glazing',
 8: 'door_entrance',
 9: 'signboard',
 10: 'awning_canopy',
 11: 'arcade_column',
 12: 'arcade_soffit',
 13: 'sidewalk_shed_scaffold',
 14: 'stoop_stair',
 15: 'wall_ledge',
 16: 'fence_railing',
 17: 'planter_container',
 18: 'tree',
 19: 'shrub_hedge',
 20: 'ground_vegetation',
 21: 'vertical_green_wall',
 22: 'bench_seating',
 23: 'pole_fixture',
 24: 'traffic_sign_signal',
 25: 'person',
 26: 'vehicle',
 27: 'sky',
 28: 'upper_building_glazing',
 29: 'traffic_cone_barrel'}
PALETTE = {0: (0, 0, 0),
 1: (128, 64, 128),
 2: (244, 35, 232),
 3: (255, 100, 100),
 4: (255, 180, 180),
 5: (70, 70, 70),
 6: (110, 80, 70),
 7: (0, 220, 255),
 8: (255, 140, 0),
 9: (255, 220, 0),
 10: (180, 100, 255),
 11: (120, 120, 160),
 12: (170, 170, 210),
 13: (0, 200, 200),
 14: (170, 90, 40),
 15: (150, 120, 80),
 16: (190, 153, 153),
 17: (180, 110, 40),
 18: (50, 130, 20),
 19: (100, 180, 40),
 20: (150, 220, 80),
 21: (0, 150, 80),
 22: (255, 80, 160),
 23: (153, 153, 153),
 24: (255, 255, 0),
 25: (220, 20, 60),
 26: (0, 0, 142),
 27: (70, 130, 180),
 28: (0, 170, 220),
 29: (255, 80, 0)}

LABEL2ID = {name: cid for cid, name in CLASS_NAMES.items()}
ID2LABEL = CLASS_NAMES.copy()
NUM_CLASSES = len(CLASS_NAMES)

# -----------------------------
# Taxonomy integrity invariants
# -----------------------------
assert NUM_CLASSES == 30
assert set(CLASS_NAMES) == set(PALETTE)
assert len(set(CLASS_NAMES.values())) == NUM_CLASSES
assert set(CLASS_NAMES.keys()) == set(range(NUM_CLASSES))
assert LABEL2ID["tree"] == 18
assert LABEL2ID["upper_building_glazing"] == 28
assert LABEL2ID["traffic_cone_barrel"] == 29
assert "tree_canopy" not in LABEL2ID
assert "tree_trunk" not in LABEL2ID
assert IGNORE_INDEX not in CLASS_NAMES

print("Source baseline:", SOURCE_BASELINE_VERSION)
print("Share package:", SHARE_VERSION)
print("Taxonomy:", TAXONOMY_VERSION)
print("Declared classes:", NUM_CLASSES)
print("Tree output policy: leaves + branches + trunk → tree")
print("Traffic-control policy: traffic_cone_barrel is a formal taxonomy class (ID 29)")
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU 未啟用。Colab: Runtime → Change runtime type → T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# AUTHORITATIVE TAXONOMY / ANNOTATION CONFIG — v1.5
#
# CLASS_NAMES + PALETTE defined above are the single source of truth.
# Config files are generated from those dictionaries so the runtime
# label map, exported taxonomy, CVAT config, and statistics cannot drift.
# ============================================================

TAXONOMY = {
    "taxonomy_version": TAXONOMY_VERSION,
    "num_classes": NUM_CLASSES,
    "ignore_index": IGNORE_INDEX,
    "classes": [
        {
            "id": cid,
            "name": CLASS_NAMES[cid],
            "rgb": list(PALETTE[cid]),
        }
        for cid in sorted(CLASS_NAMES)
    ],
}

ANNOTATION_RULES = {'other_unknown': {'include': ['Visible regions that cannot be reliably assigned to another taxonomy class after '
                               'class-specific recovery.'],
                   'exclude': ['Google Street View screenshot/UI artifacts; use IGNORE=255.']},
 'roadway': {'include': ['Motor-vehicle travel lanes', 'Visible roadway pavement', 'Bicycle-lane pavement in final output'],
             'exclude': ['Sidewalk']},
 'sidewalk': {'include': ['Pedestrian walking surface', 'Public sidewalk paving'],
              'exclude': ['Roadway', 'Building steps']},
 'bike_lane': {'include': ['Internal compatibility class only; remapped to roadway before final output'],
               'exclude': ['User-facing final statistics and legend']},
 'curb_edge': {'include': ['Visible curb face', 'Raised transition between roadway and pedestrian surface'],
               'exclude': ['Entire sidewalk']},
 'upper_building_facade': {'include': ['Opaque building facade above ground-floor interface zone'],
                           'exclude': ['Upper glazing', 'Ground-floor storefront']},
 'ground_floor_solid_facade': {'include': ['Opaque ground-floor wall or storefront facade'],
                               'exclude': ['Glazing', 'Door', 'Signboard']},
 'ground_floor_glazing': {'include': ['Ground-floor storefront glass',
                                      'Display windows',
                                      'Transparent/reflective ground-floor glazing'],
                          'exclude': ['Upper-floor glazing']},
 'door_entrance': {'include': ['Building entrance door', 'Store entrance', 'Doorway opening'], 'exclude': ['Windows']},
 'signboard': {'include': ['Storefront signage', 'Business fascia/projecting signs'], 'exclude': ['Traffic signs']},
 'awning_canopy': {'include': ['Storefront awning', 'Entrance canopy', 'Fabric entrance canopy', 'Residential entrance awning'],
                   'exclude': ['Construction sidewalk shed', 'Tree']},
 'arcade_column': {'include': ['Columns supporting a building-integrated covered pedestrian arcade'],
                   'exclude': ['Construction scaffold poles', 'Streetlight poles']},
 'arcade_soffit': {'include': ['Underside/ceiling of a building-integrated arcade'],
                   'exclude': ['Construction sidewalk shed']},
 'sidewalk_shed_scaffold': {'include': ['Temporary construction sidewalk shed',
                                        'Scaffolding structure/protective overhead shed'],
                            'exclude': ['Permanent arcade', 'Glass curtain wall', 'Glazed facade']},
 'stoop_stair': {'include': ['Exterior entrance steps', 'Stoop'], 'exclude': ['Ordinary sidewalk']},
 'wall_ledge': {'include': ['Low wall', 'Raised ledge', 'Residential perimeter wall', 'Community boundary wall', 'Masonry garden wall'], 'exclude': ['Openwork fence', 'Planter vegetation', 'Building facade']},
 'fence_railing': {'include': ['Park perimeter fence',
                               'Metal fence',
                               'Iron fence',
                               'Pedestrian railing',
                               'Openwork barrier'],
                   'exclude': ['Solid wall', 'Scaffolding', 'Building curtain wall']},
 'planter_container': {'include': ['Built planter box', 'Raised planter', 'Plant container'],
                       'exclude': ['Vegetation inside planter']},
 'tree': {'include': ['Tree leaves', 'Tree crown', 'Visible tree branches', 'Bare branches', 'Main visible tree trunk'],
          'exclude': ['Shrub/hedge', 'Ground vegetation', 'Vertical green wall']},
 'shrub_hedge': {'include': ['Shrub', 'Hedge', 'Dense eye-level woody vegetation'],
                 'exclude': ['Tree', 'Ground grass']},
 'ground_vegetation': {'include': ['Grass', 'Low groundcover vegetation'], 'exclude': ['Shrub', 'Tree']},
 'vertical_green_wall': {'include': ['Vegetation covering a vertical facade/support'],
                         'exclude': ['Tree overlapping wall in perspective']},
 'bench_seating': {'include': ['Bench', 'Fixed public seating'], 'exclude': ['Low wall']},
 'pole_fixture': {'include': ['Streetlight pole', 'Utility pole', 'Fixed vertical street pole'],
                  'exclude': ['Tree trunk', 'Scaffolding pole', 'Google Street View UI artifact']},
 'traffic_sign_signal': {'include': ['Traffic sign', 'Traffic signal'], 'exclude': ['Commercial signboard']},
 'person': {'include': ['Pedestrian', 'Visible person'], 'exclude': []},
 'vehicle': {'include': ['Car', 'Truck', 'Bus', 'Motorcycle', 'Bicycle as vehicle object'], 'exclude': []},
 'sky': {'include': ['Visible sky'], 'exclude': ['Sky reflection in glazing']},
 'upper_building_glazing': {'include': ['Upper-floor windows',
                                        'Curtain-wall glazing',
                                        'Glazed building surfaces above ground-floor interface'],
                            'exclude': ['Ground-floor storefront glazing', 'Vehicle glass']},
 'traffic_cone_barrel': {'include': ['Traffic cone', 'Construction cone', 'Traffic barrel', 'Construction barrel/drum'],
                         'exclude': ['Traffic sign/signal', 'Ordinary street furniture', 'Planter container']}}

BASELINE_SETTINGS = {'packaging_version': 'v1.5.7-roadway-precedence-cleanup',
 'algorithm_baseline': 'hybrid_v0.10_plus_targeted_v1.3_refinements',
 'baseline_status': 'SHARE_READY_SINGLE_IMAGE',
 'taxonomy_version': 'street_interface_v1.5.7',
 'models': {'full_scene_ade': 'nvidia/segformer-b5-finetuned-ade-640-640',
            'full_scene_mapillary': 'facebook/mask2former-swin-large-mapillary-vistas-semantic',
            'detector': 'IDEA-Research/grounding-dino-base',
            'mask_refiner': 'facebook/sam2.1-hiera-small'},
 'detail_detection_thresholds': {'ground_floor_glazing': 0.2,
                                 'door_entrance': 0.21,
                                 'signboard': 0.24,
                                 'awning_canopy': 0.25,
                                 'sidewalk_shed_scaffold': 0.28,
                                 'stoop_stair': 0.22,
                                 'fence_railing': 0.18,
                                 'planter_container': 0.22,
                                 'bench_seating': 0.24,
                                 'person': 0.27,
                                 'vehicle': 0.28,
                                 'traffic_cone_barrel': 0.18},
 'notes': ['Google Street View UI residuals remain IGNORE=255 and are excluded from measurement statistics.',
           'Fence recovery is class-specific and primarily targets other_unknown pixels.',
           'Glass-dominant vertical building components are vetoed from scaffold.',
           'Tree component prompts are detection-only; taxonomy output is one tree class.',
           'traffic_cone_barrel is a declared taxonomy class from initialization; it is not dynamically appended at '
           'runtime.',
           'other_unknown is retained as an epistemic fallback and is not generically filled.'],
 'declared_num_classes': 30}

CVAT_LABELS = [
    {
        "name": CLASS_NAMES[cid],
        "color": "#%02x%02x%02x" % PALETTE[cid],
        "attributes": [],
    }
    for cid in sorted(CLASS_NAMES)
]

# -----------------------------
# Cross-config audit
# -----------------------------
assert TAXONOMY["num_classes"] == NUM_CLASSES == 30
assert [x["id"] for x in TAXONOMY["classes"]] == list(range(NUM_CLASSES))
assert [x["name"] for x in TAXONOMY["classes"]] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert [x["name"] for x in CVAT_LABELS] == [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
assert set(ANNOTATION_RULES) == set(CLASS_NAMES.values())
assert "tree_canopy" not in ANNOTATION_RULES
assert "tree_trunk" not in ANNOTATION_RULES

for name, obj in [
    ("taxonomy_v1_5.json", TAXONOMY),
    ("annotation_rules_v1_5.json", ANNOTATION_RULES),
    ("baseline_settings_v1_5.json", BASELINE_SETTINGS),
    ("cvat_labels_v1_5.json", CVAT_LABELS),
]:
    with open(CONFIG_DIR / name, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

print("✓ v1.4 config exported to", CONFIG_DIR)
print("✓ Runtime taxonomy = exported taxonomy = CVAT taxonomy")
print("✓ 30 declared classes, including traffic_cone_barrel")
print("✓ Tree taxonomy is one class: tree")


## Input
Upload exactly one street-view image. The filename stem becomes the test ID.


In [ ]:
from google.colab import files
import io

print("Upload exactly ONE street-view image for this run.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Expected exactly 1 image, received {len(uploaded)}")

filename = next(iter(uploaded.keys()))
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
W, H = image.size

# Stable test ID: filename stem by default.
TEST_ID = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(filename).stem).upper()

print("Current test:", TEST_ID)
print("Image:", filename)
print("Size:", image.size)
display(image)


## Runtime helpers


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
import cv2

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

assert "upper_building_glazing" in LABEL2ID

W, H = image.size

print("Device:", device)
print("Classes:", len(CLASS_NAMES))
print("Image:", W, "x", H)

def render_taxonomy(label_map):

    rgb = np.zeros(
        (
            label_map.shape[0],
            label_map.shape[1],
            3
        ),
        dtype=np.uint8
    )

    for class_id, color in PALETTE.items():

        rgb[
            label_map == class_id
        ] = color

    return rgb


UNKNOWN_ID = LABEL2ID["other_unknown"]

print("✓ renderer ready")

for name in [
    "ade_model",
    "map_model",
    "grounding_model",
    "sam_model_new",
    "tree_detector",
    "tree_sam"
]:
    try:
        del globals()[name]
    except:
        pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ GPU cleared")


## ADE20K full-scene pass


In [ ]:
import torch.nn.functional as F

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

ADE_MODEL_ID = (
    "nvidia/segformer-b5-finetuned-ade-640-640"
)

print("Loading ADE20K...")

ade_processor = (
    SegformerImageProcessor
    .from_pretrained(
        ADE_MODEL_ID
    )
)

ade_model = (
    SegformerForSemanticSegmentation
    .from_pretrained(
        ADE_MODEL_ID
    )
    .to(device)
)

ade_model.eval()

print("✓ ADE loaded")

@torch.no_grad()
def run_ade(image):

    W, H = image.size

    inputs = ade_processor(
        images=image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    outputs = ade_model(**inputs)

    logits = F.interpolate(
        outputs.logits,
        size=(H, W),
        mode="bilinear",
        align_corners=False
    )

    probs = torch.softmax(
        logits,
        dim=1
    )

    confidence, prediction = probs.max(
        dim=1
    )

    return (
        prediction[0]
        .cpu()
        .numpy()
        .astype(np.int32),

        confidence[0]
        .cpu()
        .numpy()
        .astype(np.float32)
    )


ade_pred, ade_confidence = run_ade(
    image
)

ADE_ID2LABEL = {
    int(k): str(v).lower()
    for k, v
    in ade_model.config.id2label.items()
}

print("✓ ADE inference complete")

def ade_ids_fuzzy(
    include_terms,
    exclude_terms=None
):

    if exclude_terms is None:
        exclude_terms = []

    found = []

    for class_id, name in ADE_ID2LABEL.items():

        included = any(
            term.lower() in name
            for term in include_terms
        )

        excluded = any(
            term.lower() in name
            for term in exclude_terms
        )

        if included and not excluded:
            found.append(class_id)

    return found


def ade_mask_fuzzy(
    include_terms,
    exclude_terms=None
):

    ids = ade_ids_fuzzy(
        include_terms,
        exclude_terms
    )

    if not ids:

        return np.zeros(
            (H, W),
            dtype=bool
        )

    return np.isin(
        ade_pred,
        ids
    )


ADE_GLASS_V2 = ade_mask_fuzzy([
    "window",
    "glass"
])

ADE_ROAD_V2 = ade_mask_fuzzy([
    "road"
])

ADE_SIDEWALK_V2 = ade_mask_fuzzy([
    "sidewalk",
    "path"
])

ADE_BUILDING_V2 = ade_mask_fuzzy([
    "building",
    "house",
    "skyscraper"
])

ADE_SKY_V2 = ade_mask_fuzzy([
    "sky"
])

ADE_TREE_V2 = ade_mask_fuzzy([
    "tree"
])

ADE_PLANT_V2 = ade_mask_fuzzy([
    "plant",
    "grass",
    "flower"
])

ADE_FENCE_V3 = ade_mask_fuzzy([
    "fence",
    "railing"
])


print("ADE glass:", int(ADE_GLASS_V2.sum()))
print("ADE building:", int(ADE_BUILDING_V2.sum()))
print("ADE trees:", int(ADE_TREE_V2.sum()))
print("ADE fence/railing:", int(ADE_FENCE_V3.sum()))

del ade_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ ADE model released")


## Mapillary full-scene pass


In [ ]:
from transformers import (
    AutoImageProcessor,
    Mask2FormerForUniversalSegmentation
)

MAPILLARY_MODEL_ID = (
    "facebook/"
    "mask2former-swin-large-"
    "mapillary-vistas-semantic"
)

print("Loading Mapillary...")

map_processor = (
    AutoImageProcessor
    .from_pretrained(
        MAPILLARY_MODEL_ID
    )
)

map_model = (
    Mask2FormerForUniversalSegmentation
    .from_pretrained(
        MAPILLARY_MODEL_ID
    )
    .to(device)
)

map_model.eval()

print("✓ Mapillary loaded")

inputs = map_processor(
    images=image,
    return_tensors="pt"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():

    outputs = map_model(
        **inputs
    )


map_pred = (
    map_processor
    .post_process_semantic_segmentation(
        outputs,
        target_sizes=[
            (H, W)
        ]
    )[0]
    .cpu()
    .numpy()
    .astype(np.int32)
)


MAP_ID2LABEL = {
    int(k): str(v).lower()
    for k, v
    in map_model.config.id2label.items()
}

print("✓ Mapillary inference complete")

def mapillary_ids(
    include_terms,
    exclude_terms=None
):

    if exclude_terms is None:
        exclude_terms = []

    found = []

    for class_id, name in MAP_ID2LABEL.items():

        included = any(
            term.lower() in name
            for term in include_terms
        )

        excluded = any(
            term.lower() in name
            for term in exclude_terms
        )

        if included and not excluded:
            found.append(class_id)

    return found


def mapillary_mask(
    include_terms,
    exclude_terms=None
):

    ids = mapillary_ids(
        include_terms,
        exclude_terms
    )

    if not ids:

        return np.zeros(
            (H, W),
            dtype=bool
        )

    return np.isin(
        map_pred,
        ids
    )


MAP_ROAD = mapillary_mask(
    ["road"],
    [
        "marking",
        "bike lane",
        "service lane"
    ]
)

MAP_ROAD_MARKINGS = mapillary_mask(
    ["marking"]
)

MAP_SIDEWALK = mapillary_mask(
    ["sidewalk"]
)

MAP_BIKE_LANE = mapillary_mask(
    ["bike lane"]
)

MAP_CURB = mapillary_mask(
    ["curb"]
)

MAP_BUILDING = mapillary_mask(
    ["building"]
)

MAP_SKY = mapillary_mask(
    ["sky"]
)

MAP_FENCE = mapillary_mask(
    ["fence"]
)

MAP_WINDOW = mapillary_mask(
    ["window", "glass"]
)


print("Road:", int(MAP_ROAD.sum()))
print("Sidewalk:", int(MAP_SIDEWALK.sum()))
print("Bike lane:", int(MAP_BIKE_LANE.sum()))
print("Building:", int(MAP_BUILDING.sum()))
print("Sky:", int(MAP_SKY.sum()))
print("Fence:", int(MAP_FENCE.sum()))
print("Window/glass:", int(MAP_WINDOW.sum()))

final_test = np.full(
    (H, W),
    UNKNOWN_ID,
    dtype=np.uint8
)


# Road
final_test[
    MAP_ROAD |
    MAP_ROAD_MARKINGS
] = LABEL2ID["roadway"]


# Bike lane overrides road
final_test[
    MAP_BIKE_LANE
] = LABEL2ID["bike_lane"]


# Sidewalk
final_test[
    MAP_SIDEWALK
] = LABEL2ID["sidewalk"]


# Curb
final_test[
    MAP_CURB
] = LABEL2ID["curb_edge"]


# Building
final_test[
    MAP_BUILDING
] = LABEL2ID[
    "upper_building_facade"
]


# Sky
final_test[
    MAP_SKY
] = LABEL2ID["sky"]


# ADE fallback
unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_ROAD_V2
] = LABEL2ID["roadway"]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_SIDEWALK_V2
] = LABEL2ID["sidewalk"]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_BUILDING_V2
] = LABEL2ID[
    "upper_building_facade"
]


unknown = (
    final_test == UNKNOWN_ID
)

final_test[
    unknown & ADE_SKY_V2
] = LABEL2ID["sky"]


# preliminary vegetation
final_test[
    ADE_TREE_V2
] = LABEL2ID[
    "tree"
]


building_context = (
    MAP_BUILDING
    |
    ADE_BUILDING_V2
)


# ADE preliminary glazing
basic_glass = (
    ADE_GLASS_V2
    &
    building_context
)

final_test[
    basic_glass
] = LABEL2ID[
    "upper_building_glazing"
]


print("✓ Full scene base created")

rgb_base = render_taxonomy(
    final_test
)

plt.figure(
    figsize=(18, 8)
)

plt.subplot(1, 2, 1)

plt.imshow(image)

plt.title(
    f"{TEST_ID} — Original"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    rgb_base
)

plt.title(
    f"{TEST_ID} — Full Scene Base"
)

plt.axis("off")


plt.tight_layout()
plt.show()

del map_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ Mapillary model released")


## Grounding DINO detections


In [ ]:
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

GROUNDING_MODEL_ID = (
    "IDEA-Research/grounding-dino-base"
)

grounding_processor = (
    AutoProcessor.from_pretrained(
        GROUNDING_MODEL_ID
    )
)

grounding_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_MODEL_ID
    )
    .to(device)
)

grounding_model.eval()

print("✓ Grounding DINO loaded")

AUTO_LABEL_QUERIES = {

    "ground_floor_glazing": [
        "storefront glass window",
        "shopfront window",
        "ground floor glass window",
        "glass storefront"
    ],

    "door_entrance": [
        "building entrance door",
        "store entrance door",
        "entrance doorway"
    ],

    "signboard": [
        "storefront business sign",
        "shop signboard",
        "commercial sign"
    ],

    "awning_canopy": [
        "storefront awning",
        "entrance awning",
        "building entrance canopy",
        "fabric entrance canopy",
        "residential entrance awning",
        "projecting entrance canopy"
    ],

    "sidewalk_shed_scaffold": [
        "construction sidewalk shed",
        "sidewalk scaffolding",
        "construction scaffolding",
        "scaffold structure"
    ],

    "stoop_stair": [
        "building entrance steps",
        "residential stoop",
        "entrance stairs"
    ],

    "wall_ledge": [
        "residential perimeter wall",
        "community boundary wall",
        "masonry boundary wall",
        "low garden wall",
        "property boundary wall",
        "stone retaining wall"
    ],

    "fence_railing": [
        "park perimeter fence",
        "black iron park fence",
        "metal park fence",
        "pedestrian railing",
        "metal railing",
        "ornamental iron fence",
        "openwork metal fence"
    ],

    "planter_container": [
        "street planter",
        "planter box",
        "plant container"
    ],

    "bench_seating": [
        "street bench",
        "outdoor bench"
    ],

    "person": [
        "pedestrian",
        "person"
    ],

    "vehicle": [
        "car",
        "truck",
        "bus",
        "bicycle"
    ]
}


CLASS_THRESHOLDS = {

    "ground_floor_glazing": 0.20,
    "door_entrance": 0.21,
    "signboard": 0.24,
    "awning_canopy": 0.18,

    "sidewalk_shed_scaffold": 0.28,

    "stoop_stair": 0.22,
    "wall_ledge": 0.18,
    "fence_railing": 0.18,
    "planter_container": 0.22,
    "bench_seating": 0.24,

    "person": 0.27,
    "vehicle": 0.28
}

print("✓ frozen detail configuration loaded")

DETAIL_CROPS = {

    "left_interface": (
        0,
        int(H * 0.22),
        int(W * 0.58),
        int(H * 0.88)
    ),

    "center_interface": (
        int(W * 0.18),
        int(H * 0.20),
        int(W * 0.82),
        int(H * 0.90)
    ),

    "right_interface": (
        int(W * 0.40),
        int(H * 0.18),
        W,
        int(H * 0.90)
    )
}

@torch.no_grad()
def detect_one_class(
    source_image,
    class_name,
    offset_x=0,
    offset_y=0
):

    phrases = AUTO_LABEL_QUERIES[
        class_name
    ]

    inputs = grounding_processor(
        images=source_image,
        text=[phrases],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=CLASS_THRESHOLDS[
                class_name
            ],
            text_threshold=0.18,
            target_sizes=[
                (
                    source_image.height,
                    source_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "class_name":
                class_name,

            "class_id":
                LABEL2ID[
                    class_name
                ],

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

DETAIL_CLASSES = [
    "ground_floor_glazing",
    "door_entrance",
    "signboard",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "stoop_stair",
    "wall_ledge",
    "fence_railing",
    "planter_container",
    "bench_seating",
    "person",
    "vehicle"
]


detail_detections_raw = []


for class_name in DETAIL_CLASSES:

    print(
        "Detail:",
        class_name
    )

    for crop_name, crop_box in (
        DETAIL_CROPS.items()
    ):

        x1, y1, x2, y2 = crop_box

        crop = image.crop(
            crop_box
        )

        detections = detect_one_class(
            crop,
            class_name,
            offset_x=x1,
            offset_y=y1
        )

        detail_detections_raw.extend(
            detections
        )


print(
    "Raw detail detections:",
    len(detail_detections_raw)
)

WINDOW_PROMPTS = [
    "apartment building window",
    "building window",
    "residential window",
    "glass window",
    "storefront window",
    "glass curtain wall"
]


FACADE_TILES = {

    "left_upper": (
        0,
        0,
        int(W * 0.60),
        int(H * 0.68)
    ),

    "center_upper": (
        int(W * 0.15),
        0,
        int(W * 0.85),
        int(H * 0.72)
    ),

    "right_upper": (
        int(W * 0.40),
        0,
        W,
        int(H * 0.70)
    )
}

@torch.no_grad()
def detect_windows(
    crop_image,
    offset_x,
    offset_y
):

    inputs = grounding_processor(
        images=crop_image,
        text=[WINDOW_PROMPTS],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=0.15,
            text_threshold=0.17,
            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "class_name":
                "building_glazing",

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

window_detections_raw = []

for tile_name, crop_box in (
    FACADE_TILES.items()
):

    print(
        "Window:",
        tile_name
    )

    x1, y1, x2, y2 = crop_box

    crop = image.crop(
        crop_box
    )

    window_detections_raw.extend(
        detect_windows(
            crop,
            x1,
            y1
        )
    )


print(
    "Raw window detections:",
    len(window_detections_raw)
)

TREE_QUERY_GROUPS = {

    "foliage": [
        "green tree foliage",
        "leafy tree crown",
        "tree leaves",
        "green leaves on a tree"
    ],

    "woody": [
        "leafless street tree",
        "bare deciduous tree branches",
        "woody tree branches",
        "street tree trunk",
        "tree trunk",
        "tree limb"
    ]
}


TREE_ROIS = {

    "left_tree": (
        0,
        0,
        int(W * 0.52),
        int(H * 0.78)
    ),

    "center_tree": (
        int(W * 0.18),
        0,
        int(W * 0.75),
        int(H * 0.75)
    ),

    "right_tree": (
        int(W * 0.45),
        0,
        W,
        int(H * 0.75)
    )
}

@torch.no_grad()
def detect_tree_component(
    crop_image,
    prompts,
    component_name,
    offset_x,
    offset_y,
    threshold
):

    inputs = grounding_processor(
        images=crop_image,
        text=[prompts],
        return_tensors="pt"
    ).to(device)

    outputs = grounding_model(
        **inputs
    )

    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=threshold,
            text_threshold=0.17,
            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]

    detections = []

    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )

        detections.append({

            "component":
                component_name,

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                x1 + offset_x,
                y1 + offset_y,
                x2 + offset_x,
                y2 + offset_y
            ]
        })

    return detections

tree_detections_raw = []

for roi_name, roi_box in TREE_ROIS.items():

    x1, y1, x2, y2 = roi_box

    crop = image.crop(
        roi_box
    )

    tree_detections_raw.extend(

        detect_tree_component(
            crop,
            TREE_QUERY_GROUPS["foliage"],
            "foliage",
            x1,
            y1,
            threshold=0.27
        )
    )

    tree_detections_raw.extend(

        detect_tree_component(
            crop,
            TREE_QUERY_GROUPS["woody"],
            "woody",
            x1,
            y1,
            threshold=0.16
        )
    )


print(
    "Raw tree detections:",
    len(tree_detections_raw)
)

from torchvision.ops import nms


def nms_group(
    detections,
    key_name,
    iou=0.42
):

    if not detections:
        return []

    result = []

    groups = sorted(
        set(
            d[key_name]
            for d in detections
        )
    )

    for group in groups:

        subset = [
            d
            for d in detections
            if d[key_name] == group
        ]

        boxes = torch.tensor(
            [
                d["box"]
                for d in subset
            ],
            dtype=torch.float32
        )

        scores = torch.tensor(
            [
                d["score"]
                for d in subset
            ],
            dtype=torch.float32
        )

        keep = nms(
            boxes,
            scores,
            iou
        )

        result.extend(
            subset[i]
            for i in keep.tolist()
        )

    return result


detail_detections = nms_group(
    detail_detections_raw,
    "class_name"
)

window_detections = nms_group(
    window_detections_raw,
    "class_name"
)

tree_detections = nms_group(
    tree_detections_raw,
    "component"
)


print(
    "Detail:",
    len(detail_detections)
)

print(
    "Windows:",
    len(window_detections)
)

print(
    "Trees:",
    len(tree_detections)
)

del grounding_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✓ DINO released")


## SAM2 mask generation


In [ ]:
from transformers import (
    Sam2Processor,
    Sam2Model
)

SAM_MODEL_ID = (
    "facebook/sam2.1-hiera-small"
)

sam_processor = (
    Sam2Processor.from_pretrained(
        SAM_MODEL_ID
    )
)

sam_model = (
    Sam2Model.from_pretrained(
        SAM_MODEL_ID
    )
    .to(device)
)

sam_model.eval()

print("✓ SAM2 loaded")

@torch.no_grad()
def boxes_to_masks(
    image,
    detections
):

    if not detections:
        return []

    boxes = [
        d["box"]
        for d in detections
    ]

    inputs = (
        sam_processor(
            images=image,
            input_boxes=[boxes],
            return_tensors="pt"
        )
        .to(device)
    )

    outputs = sam_model(
        **inputs,
        multimask_output=False
    )

    masks = (
        sam_processor
        .post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"]
        )[0]
    )

    output = []

    for i, det in enumerate(
        detections
    ):

        mask = masks[i]

        while mask.ndim > 2:
            mask = mask[0]

        output.append({
            **det,
            "mask":
                mask.numpy() > 0
        })

    return output

detail_objects = boxes_to_masks(
    image,
    detail_detections
)

window_objects = boxes_to_masks(
    image,
    window_detections
)

tree_objects = boxes_to_masks(
    image,
    tree_detections
)


print(
    "Detail masks:",
    len(detail_objects)
)

print(
    "Window masks:",
    len(window_objects)
)

print(
    "Tree masks:",
    len(tree_objects)
)


## Core hybrid fusion


In [ ]:
final_v06 = final_test.copy()


INTERFACE_ANCHOR_CLASSES = {
    "ground_floor_glazing",
    "door_entrance",
    "signboard",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "stoop_stair"
}


interface_anchor = np.zeros(
    (H, W),
    dtype=bool
)


for obj in detail_objects:

    if (
        obj["class_name"]
        in INTERFACE_ANCHOR_CLASSES
        and
        obj["score"] >= 0.23
    ):

        interface_anchor |= obj[
            "mask"
        ]


kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (51, 31)
)


interface_neighborhood = (
    cv2.dilate(
        interface_anchor.astype(
            np.uint8
        ),
        kernel,
        iterations=1
    ) > 0
)


ground_floor_solid = (
    building_context
    &
    interface_neighborhood
    &
    ~ADE_GLASS_V2
)


final_v06[
    ground_floor_solid
] = LABEL2ID[
    "ground_floor_solid_facade"
]


print("✓ v0.6 interface neighborhood built")

facade_glass_aggressive = np.zeros(
    (H, W),
    dtype=bool
)


for obj in window_objects:

    gated = (
        obj["mask"]
        &
        building_context
    )

    original_area = max(
        int(
            obj["mask"].sum()
        ),
        1
    )

    agreement = (
        gated.sum()
        /
        original_area
    )

    if agreement < 0.25:
        continue

    facade_glass_aggressive |= gated


ground_glass = (
    facade_glass_aggressive
    &
    interface_neighborhood
)

upper_glass = (
    facade_glass_aggressive
    &
    ~interface_neighborhood
)


final_v06[
    ground_glass
] = LABEL2ID[
    "ground_floor_glazing"
]

final_v06[
    upper_glass
] = LABEL2ID[
    "upper_building_glazing"
]


print(
    "Aggressive upper glass:",
    upper_glass.mean() * 100
)

FUSION_MIN_SCORE = {

    "ground_floor_glazing":
        0.22,

    "door_entrance":
        0.23,

    "signboard":
        0.26,

    "awning_canopy":
        0.27,

    "sidewalk_shed_scaffold":
        0.28,

    "stoop_stair":
        0.24,

    "wall_ledge":
        0.28,

    "fence_railing":
        0.20,

    "planter_container":
        0.24,

    "bench_seating":
        0.25,

    "person":
        0.27,

    "vehicle":
        0.28
}


for obj in detail_objects:

    class_name = obj[
        "class_name"
    ]

    score = obj[
        "score"
    ]

    if (
        score
        <
        FUSION_MIN_SCORE[
            class_name
        ]
    ):
        continue

    mask = obj[
        "mask"
    ]


    if (
        class_name
        ==
        "ground_floor_glazing"
    ):

        allowed = (
            building_context
            &
            interface_neighborhood
        )


    elif class_name in {
        "door_entrance",
        "signboard",
        "awning_canopy"
    }:

        allowed = (
            building_context
            |
            interface_neighborhood
        )


    elif (
        class_name
        ==
        "sidewalk_shed_scaffold"
    ):

        allowed = (
            MAP_BUILDING
            |
            MAP_SIDEWALK
        )


    elif class_name == "wall_ledge":

        wall_sidewalk_near = cv2.dilate(
            MAP_SIDEWALK.astype(np.uint8),
            cv2.getStructuringElement(cv2.MORPH_RECT, (91, 61)),
            iterations=1
        ) > 0

        allowed = (
            building_context
            |
            wall_sidewalk_near
        )


    elif class_name == "fence_railing":

        # v1.3: park perimeter fences are often next to open space,
        # not directly on a building footprint.
        fence_sidewalk_near = cv2.dilate(
            MAP_SIDEWALK.astype(np.uint8),
            cv2.getStructuringElement(cv2.MORPH_RECT, (81, 51)),
            iterations=1
        ) > 0

        allowed = (
            MAP_FENCE
            | ADE_FENCE_V3
            | fence_sidewalk_near
            | building_context
        )

    elif class_name in {
        "stoop_stair",
        "planter_container",
        "bench_seating"
    }:

        allowed = (
            MAP_SIDEWALK
            |
            building_context
        )


    else:

        allowed = np.ones(
            (H, W),
            dtype=bool
        )


    gated = (
        mask
        &
        allowed
    )


    original_area = max(
        int(mask.sum()),
        1
    )


    if (
        gated.sum()
        /
        original_area
        <
        0.12
    ):
        continue


    final_v06[
        gated
    ] = LABEL2ID[
        class_name
    ]


print("✓ v0.6-style detail fusion complete")

dynamic_mask = np.isin(
    final_v06,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"]
    ]
)


final_v06[
    MAP_ROAD
    &
    ~dynamic_mask
] = LABEL2ID[
    "roadway"
]


final_v06[
    MAP_BIKE_LANE
    &
    ~dynamic_mask
] = LABEL2ID[
    "bike_lane"
]


final_v06[
    MAP_CURB
    &
    ~dynamic_mask
] = LABEL2ID[
    "curb_edge"
]


print("✓ v0.6 street surfaces protected")

def mask_geometry(mask):

    ys, xs = np.where(mask)

    if len(xs) == 0:

        return {
            "area_ratio": 0,
            "cx": 0,
            "cy": 0
        }

    return {

        "area_ratio":
            float(mask.mean()),

        "cx":
            float(xs.mean() / W),

        "cy":
            float(ys.mean() / H)
    }


accepted_windows = []


for obj in window_objects:

    geometry = mask_geometry(
        obj["mask"]
    )

    # Reject huge window hallucinations
    if (
        geometry["area_ratio"]
        >
        0.015
    ):
        continue

    if (
        geometry["cy"]
        >
        0.80
    ):
        continue

    gated = (
        obj["mask"]
        &
        building_context
    )

    original_area = max(
        int(
            obj["mask"].sum()
        ),
        1
    )

    agreement = (
        gated.sum()
        /
        original_area
    )

    if agreement < 0.50:
        continue

    accepted_windows.append({
        **obj,
        "mask":
            gated
    })


print(
    "Windows:",
    len(window_objects),
    "→ accepted:",
    len(accepted_windows)
)

final_v09 = final_v06.copy()


UPPER_FACADE_ID = LABEL2ID[
    "upper_building_facade"
]

UPPER_GLASS_ID = LABEL2ID[
    "upper_building_glazing"
]


precise_window_mask = np.zeros(
    (H, W),
    dtype=bool
)


for obj in accepted_windows:

    precise_window_mask |= (
        obj["mask"]
    )

SCAFFOLD_ID = LABEL2ID[
    "sidewalk_shed_scaffold"
]


scaffold_core = (
    final_v06
    ==
    SCAFFOLD_ID
)


def shift_mask(mask, dy=0, dx=0):

    out = np.zeros_like(
        mask,
        dtype=bool
    )

    y0 = max(0, -dy)
    y1 = mask.shape[0] - max(0, dy)

    x0 = max(0, -dx)
    x1 = mask.shape[1] - max(0, dx)

    yd0 = max(0, dy)
    xd0 = max(0, dx)

    if y1 > y0 and x1 > x0:

        out[
            yd0:yd0 + (y1-y0),
            xd0:xd0 + (x1-x0)
        ] = mask[
            y0:y1,
            x0:x1
        ]

    return out


under_scaffold_keep = np.zeros(
    (H, W),
    dtype=bool
)


for dy in [
    0,
    20,
    40,
    60,
    80,
    100
]:

    under_scaffold_keep |= (
        shift_mask(
            scaffold_core,
            dy=dy
        )
    )


under_scaffold_keep = (
    cv2.dilate(
        under_scaffold_keep.astype(
            np.uint8
        ),
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (71, 29)
        ),
        iterations=1
    ) > 0
)


print(
    "Protected under-scaffold:",
    int(
        under_scaffold_keep.sum()
    )
)

upper_precision_zone = np.zeros(
    (H, W),
    dtype=bool
)

upper_precision_zone[
    :int(H * 0.55),
    :
] = True


upper_precision_zone &= (
    building_context
)

upper_precision_zone &= (
    ~under_scaffold_keep
)


# Reset building pixels in upper precision zone
# to solid façade first
final_v09[
    upper_precision_zone
] = UPPER_FACADE_ID


# Then install only accepted windows
final_v09[
    precise_window_mask
    &
    upper_precision_zone
] = UPPER_GLASS_ID


print(
    "✓ upper building precision transplanted"
)

STOOP_ID = LABEL2ID[
    "stoop_stair"
]


# ============================================================
# v1.4 STOOP PRECISION
#
# A sidewalk pixel does NOT become a stoop merely because it is
# on / near the sidewalk.
#
# Keep stoop only when the component is:
# - close to a reliable entrance / doorway anchor
# - compact enough to behave like entrance steps
# - located in the lower / interface portion of the scene
#
# Rejected false stoop is restored to its underlying semantic
# surface, with sidewalk taking priority when Mapillary says
# sidewalk.
# ============================================================

door_anchor = np.zeros(
    (H, W),
    dtype=bool
)

for obj in detail_objects:

    if (
        obj["class_name"] == "door_entrance"
        and
        obj["score"] >= 0.23
    ):

        geom = mask_geometry(
            obj["mask"]
        )

        if geom["area_ratio"] <= 0.035:
            door_anchor |= obj["mask"]


entrance_neighborhood = (
    cv2.dilate(
        door_anchor.astype(np.uint8),
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (61, 41)
        ),
        iterations=1
    ) > 0
)


current_stoop = (
    final_v09 == STOOP_ID
)


num_stoop, stoop_cc, stoop_stats, _ = (
    cv2.connectedComponentsWithStats(
        current_stoop.astype(np.uint8),
        connectivity=8
    )
)


valid_stoop_v14 = np.zeros(
    (H, W),
    dtype=bool
)

rejected_stoop_v14 = np.zeros(
    (H, W),
    dtype=bool
)

stoop_audit_rows = []


for cid in range(
    1,
    num_stoop
):

    component = (
        stoop_cc == cid
    )

    area = int(
        component.sum()
    )

    if area == 0:
        continue

    x = int(
        stoop_stats[
            cid,
            cv2.CC_STAT_LEFT
        ]
    )

    y = int(
        stoop_stats[
            cid,
            cv2.CC_STAT_TOP
        ]
    )

    w = int(
        stoop_stats[
            cid,
            cv2.CC_STAT_WIDTH
        ]
    )

    h = int(
        stoop_stats[
            cid,
            cv2.CC_STAT_HEIGHT
        ]
    )

    area_ratio = (
        area
        /
        (H * W)
    )

    width_ratio = (
        w
        /
        W
    )

    center_y = (
        y + h / 2
    ) / H

    entrance_overlap = float(
        entrance_neighborhood[
            component
        ].mean()
    )

    sidewalk_overlap = float(
        MAP_SIDEWALK[
            component
        ].mean()
    )

    compact = (
        area_ratio <= 0.022
        and
        width_ratio <= 0.28
    )

    lower_interface = (
        center_y >= 0.38
    )

    entrance_supported = (
        entrance_overlap >= 0.08
    )

    keep = (
        compact
        and
        lower_interface
        and
        entrance_supported
    )

    if keep:

        valid_stoop_v14[
            component
        ] = True

    else:

        rejected_stoop_v14[
            component
        ] = True

    stoop_audit_rows.append({
        "component": cid,
        "pixels": area,
        "area_ratio": area_ratio,
        "width_ratio": width_ratio,
        "center_y": center_y,
        "entrance_overlap": entrance_overlap,
        "sidewalk_overlap": sidewalk_overlap,
        "kept_as_stoop": keep
    })


# Restore false stoop to underlying semantics.
restore_stoop_sidewalk = (
    rejected_stoop_v14
    &
    MAP_SIDEWALK
)

restore_stoop_building = (
    rejected_stoop_v14
    &
    ~MAP_SIDEWALK
    &
    building_context
)

restore_stoop_unknown = (
    rejected_stoop_v14
    &
    ~MAP_SIDEWALK
    &
    ~building_context
)


final_v09[
    restore_stoop_sidewalk
] = LABEL2ID[
    "sidewalk"
]


final_v09[
    restore_stoop_building
] = LABEL2ID[
    "ground_floor_solid_facade"
]


final_v09[
    restore_stoop_unknown
] = UNKNOWN_ID


print(
    "v1.4 stoop kept:",
    int(
        valid_stoop_v14.sum()
    )
)

print(
    "v1.4 false stoop rejected:",
    int(
        rejected_stoop_v14.sum()
    )
)

if stoop_audit_rows:

    display(
        pd.DataFrame(
            stoop_audit_rows
        )
    )



# ============================================================
# v1.4 SEMANTIC BACKBONE COMMIT
#
# STOP HERE.
#
# Legacy v0.10 used DINO/SAM tree proposals to write broad
# foliage / woody masks directly into the label map.
#
# v1.4 intentionally does NOT do that.
#
# final_v09 is preserved as the authoritative semantic backbone.
# ============================================================

semantic_backbone = (
    final_v09.copy()
)

final_v010 = (
    semantic_backbone.copy()
)


# ------------------------------------------------------------
# Backbone protection snapshots
# ------------------------------------------------------------

BACKBONE_PROTECTED_CLASS_NAMES = [
    "roadway",
    "sidewalk",
    "curb_edge",
    "upper_building_facade",
    "upper_building_glazing",
    "ground_floor_solid_facade",
    "ground_floor_glazing",
    "sky",
]


BACKBONE_PROTECTED_IDS = [
    LABEL2ID[name]
    for name in
    BACKBONE_PROTECTED_CLASS_NAMES
]


backbone_protected_mask = np.isin(
    semantic_backbone,
    BACKBONE_PROTECTED_IDS
)


backbone_glazing_mask = np.isin(
    semantic_backbone,
    [
        LABEL2ID[
            "upper_building_glazing"
        ],
        LABEL2ID[
            "ground_floor_glazing"
        ]
    ]
)


backbone_sidewalk_mask = (
    semantic_backbone
    ==
    LABEL2ID[
        "sidewalk"
    ]
)


backbone_tree_mask = (
    semantic_backbone
    ==
    LABEL2ID[
        "tree"
    ]
)


print(
    "✓ v0.9 semantic backbone committed"
)

print(
    "Backbone glazing pixels:",
    int(
        backbone_glazing_mask.sum()
    )
)

print(
    "Backbone sidewalk pixels:",
    int(
        backbone_sidewalk_mask.sum()
    )
)

print(
    "Backbone tree pixels:",
    int(
        backbone_tree_mask.sum()
    )
)


# ------------------------------------------------------------
# Visual checkpoint
# ------------------------------------------------------------

rgb_backbone = render_taxonomy(
    semantic_backbone
)

original_np = np.array(
    image
)

overlay_backbone = (
    original_np.copy()
)

classified_backbone = (
    semantic_backbone
    !=
    UNKNOWN_ID
)

overlay_backbone[
    classified_backbone
] = (
    original_np[
        classified_backbone
    ] * 0.42
    +
    rgb_backbone[
        classified_backbone
    ] * 0.58
).astype(
    np.uint8
)


plt.figure(
    figsize=(18, 8)
)

plt.subplot(
    1,
    2,
    1
)

plt.imshow(
    image
)

plt.title(
    f"{TEST_ID} — Original"
)

plt.axis(
    "off"
)


plt.subplot(
    1,
    2,
    2
)

plt.imshow(
    overlay_backbone
)

plt.title(
    f"{TEST_ID} — v1.4 Protected Semantic Backbone"
)

plt.axis(
    "off"
)


plt.tight_layout()
plt.show()


## Stable precision refinements

The following cells are retained because they are part of the latest stable single-image result.  
They refine the base hybrid output without using the later experimental unknown-recovery or batch-replay logic.


In [ ]:
# ============================================================
# v1.4 WORKING STATE
#
# All later refinements start from the v0.9 semantic backbone.
# There is no v0.10 broad-tree rewrite in this version.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import gc


final_working = (
    semantic_backbone.copy()
)


TRAFFIC_CONTROL_NAME = (
    "traffic_cone_barrel"
)

TRAFFIC_CONTROL_ID = LABEL2ID[
    TRAFFIC_CONTROL_NAME
]


assert TRAFFIC_CONTROL_ID == 29
assert NUM_CLASSES == 30


print(
    "✓ v1.4 starts from semantic_backbone"
)

print(
    "Traffic-control class ID:",
    TRAFFIC_CONTROL_ID
)

print(
    "Declared taxonomy classes:",
    NUM_CLASSES
)


In [ ]:
# ============================================================
# TREE POLICY — v1.4 BACKBONE PROTECTED
#
# IMPORTANT:
#
# The old v0.10 behavior:
#
#     final[woody_mask]   = tree
#     final[foliage_mask] = tree
#
# allowed a broad DINO/SAM proposal to overwrite an already
# correct building façade / glazing result.
#
# v1.4 removes that overwrite path completely.
#
# Tree is taken from the v0.9 semantic backbone.
#
# DINO/SAM tree proposals are retained only as diagnostics for
# future validation. They have ZERO write permission here.
# ============================================================

TREE_ID = LABEL2ID[
    "tree"
]

ROAD_ID = LABEL2ID[
    "roadway"
]

SIDEWALK_ID = LABEL2ID[
    "sidewalk"
]

CURB_ID = LABEL2ID[
    "curb_edge"
]

UPPER_FACADE_ID = LABEL2ID[
    "upper_building_facade"
]

GROUND_SOLID_ID = LABEL2ID[
    "ground_floor_solid_facade"
]

UPPER_GLASS_ID = LABEL2ID[
    "upper_building_glazing"
]

GROUND_GLASS_ID = LABEL2ID[
    "ground_floor_glazing"
]


# Diagnostic proposal masks only.
raw_foliage_diagnostic = np.zeros(
    (H, W),
    dtype=bool
)

raw_woody_diagnostic = np.zeros(
    (H, W),
    dtype=bool
)


for obj in tree_objects:

    if (
        obj["component"] == "foliage"
        and
        obj["score"] >= 0.27
    ):

        raw_foliage_diagnostic |= (
            obj["mask"]
        )

    elif (
        obj["component"] == "woody"
        and
        obj["score"] >= 0.18
    ):

        raw_woody_diagnostic |= (
            obj["mask"]
        )


tree_proposal_on_glazing = (
    (
        raw_foliage_diagnostic
        |
        raw_woody_diagnostic
    )
    &
    backbone_glazing_mask
)


print(
    "Backbone tree %:",
    backbone_tree_mask.mean()
    *
    100
)

print(
    "DINO/SAM foliage proposal % (diagnostic only):",
    raw_foliage_diagnostic.mean()
    *
    100
)

print(
    "DINO/SAM woody proposal % (diagnostic only):",
    raw_woody_diagnostic.mean()
    *
    100
)

print(
    "Tree proposal pixels that WOULD have hit glazing:",
    int(
        tree_proposal_on_glazing.sum()
    )
)

print(
    "✓ Tree proposals have no label-write permission in v1.4"
)


In [ ]:
# ============================================================
# TRAFFIC CONE / CONSTRUCTION BARREL DETECTION
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    Sam2Processor,
    Sam2Model
)

from torchvision.ops import nms


# ------------------------------------------------------------
# Free SAM temporarily before loading DINO
# to avoid T4 VRAM overflow
# ------------------------------------------------------------

if (
    "sam_model"
    in globals()
):

    try:
        del sam_model
    except:
        pass


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load Grounding DINO
# ------------------------------------------------------------

GROUNDING_MODEL_ID = (
    "IDEA-Research/grounding-dino-base"
)


if (
    "grounding_processor"
    not in globals()
):

    grounding_processor = (
        AutoProcessor.from_pretrained(
            GROUNDING_MODEL_ID
        )
    )


grounding_model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_MODEL_ID
    )
    .to(device)
)


grounding_model.eval()

print(
    "✓ Grounding DINO loaded for traffic-control detection"
)


TRAFFIC_CONTROL_PROMPTS = [

    "orange traffic cone",
    "road traffic cone",
    "construction traffic cone",
    "construction cone",

    "orange construction barrel",
    "traffic barrel",
    "construction barrel",
    "construction drum"
]


# ------------------------------------------------------------
# Traffic-control ROIs
#
# Focus on street / eye-level zone.
# Overlapping crops help small cones.
# ------------------------------------------------------------

TRAFFIC_ROIS = {

    "left_street": (
        0,
        int(
            H * 0.28
        ),
        int(
            W * 0.58
        ),
        int(
            H * 0.90
        )
    ),

    "center_street": (
        int(
            W * 0.18
        ),
        int(
            H * 0.26
        ),
        int(
            W * 0.84
        ),
        int(
            H * 0.90
        )
    ),

    "right_street": (
        int(
            W * 0.42
        ),
        int(
            H * 0.28
        ),
        W,
        int(
            H * 0.90
        )
    )
}


@torch.no_grad()
def detect_traffic_control(
    crop_image,
    offset_x,
    offset_y
):

    inputs = (
        grounding_processor(
            images=crop_image,
            text=[
                TRAFFIC_CONTROL_PROMPTS
            ],
            return_tensors="pt"
        )
        .to(device)
    )


    outputs = grounding_model(
        **inputs
    )


    results = (
        grounding_processor
        .post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,

            # Cones are small,
            # therefore slightly lower threshold
            threshold=0.18,

            text_threshold=0.16,

            target_sizes=[
                (
                    crop_image.height,
                    crop_image.width
                )
            ]
        )
    )[0]


    detections = []


    for box, score in zip(
        results["boxes"],
        results["scores"]
    ):

        x1, y1, x2, y2 = (
            box.detach()
            .cpu()
            .tolist()
        )


        x1 += offset_x
        x2 += offset_x

        y1 += offset_y
        y2 += offset_y


        box_w = max(
            x2 - x1,
            1
        )

        box_h = max(
            y2 - y1,
            1
        )


        box_area_ratio = (
            box_w
            *
            box_h
            /
            (
                W
                *
                H
            )
        )


        center_y = (
            (
                y1 + y2
            )
            /
            2
            /
            H
        )


        # Reject giant hallucinations

        if (
            box_area_ratio
            >
            0.025
        ):
            continue


        # Reject extremely tiny noise

        if (
            box_area_ratio
            <
            0.00001
        ):
            continue


        # Traffic control should be roughly
        # in street / pedestrian zone

        if not (
            0.28
            <=
            center_y
            <=
            0.92
        ):
            continue


        detections.append({

            "class_name":
                TRAFFIC_CONTROL_NAME,

            "class_id":
                TRAFFIC_CONTROL_ID,

            "score":
                float(
                    score.detach()
                    .cpu()
                    .item()
                ),

            "box": [
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ]
        })


    return detections


traffic_detections_raw = []


for (
    roi_name,
    roi_box
) in TRAFFIC_ROIS.items():

    x1, y1, x2, y2 = (
        roi_box
    )

    crop = image.crop(
        roi_box
    )

    traffic_detections_raw.extend(

        detect_traffic_control(
            crop,
            x1,
            y1
        )
    )


print(
    "Raw traffic-control detections:",
    len(
        traffic_detections_raw
    )
)


# ------------------------------------------------------------
# NMS
# ------------------------------------------------------------

if (
    len(
        traffic_detections_raw
    )
    >
    0
):

    boxes = torch.tensor(
        [
            d["box"]
            for d
            in traffic_detections_raw
        ],
        dtype=torch.float32
    )


    scores = torch.tensor(
        [
            d["score"]
            for d
            in traffic_detections_raw
        ],
        dtype=torch.float32
    )


    keep = nms(
        boxes,
        scores,
        0.35
    )


    traffic_detections = [

        traffic_detections_raw[
            i
        ]

        for i in keep.tolist()
    ]

else:

    traffic_detections = []


print(
    "After NMS:",
    len(
        traffic_detections
    )
)


# ------------------------------------------------------------
# Release DINO before SAM
# ------------------------------------------------------------

del grounding_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# Load / reload SAM2
# ------------------------------------------------------------

SAM_MODEL_ID = (
    "facebook/sam2.1-hiera-small"
)


sam_processor = (
    Sam2Processor.from_pretrained(
        SAM_MODEL_ID
    )
)


sam_model = (
    Sam2Model.from_pretrained(
        SAM_MODEL_ID
    )
    .to(device)
)


sam_model.eval()

print(
    "✓ SAM2 loaded for traffic-control masks"
)


traffic_objects = (
    boxes_to_masks(
        image,
        traffic_detections
    )
)


print(
    "Traffic-control SAM masks:",
    len(
        traffic_objects
    )
)


In [ ]:
# ============================================================
# TRAFFIC CONTROL QUALITY GATE
# ============================================================


# ------------------------------------------------------------
# Check whether Mapillary itself has a cone-like class
# ------------------------------------------------------------

traffic_mapillary_ids = (
    mapillary_ids(
        [
            "traffic cone",
            "traffic-cone",
            "cone"
        ],
        [
            "traffic sign"
        ]
    )
)


print(
    "Mapillary cone-like classes:"
)


if traffic_mapillary_ids:

    for cid in traffic_mapillary_ids:

        print(
            cid,
            MAP_ID2LABEL[
                cid
            ]
        )

else:

    print(
        "None — DINO/SAM + scene logic will be used."
    )


if traffic_mapillary_ids:

    MAP_TRAFFIC_CONTROL = (
        np.isin(
            map_pred,
            traffic_mapillary_ids
        )
    )

else:

    MAP_TRAFFIC_CONTROL = np.zeros(
        (
            H,
            W
        ),
        dtype=bool
    )


# ------------------------------------------------------------
# Street-surface context
# ------------------------------------------------------------

street_surface = (
    MAP_ROAD
    |
    MAP_SIDEWALK
    |
    MAP_CURB
)


surface_kernel = (
    cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            31,
            31
        )
    )
)


street_surface_support = (
    cv2.dilate(
        street_surface.astype(
            np.uint8
        ),
        surface_kernel,
        iterations=1
    )
    >
    0
)


# ------------------------------------------------------------
# Orange color evidence
#
# Used as supporting evidence,
# not as the segmentation itself.
# ------------------------------------------------------------

image_np = np.array(
    image
)


hsv = (
    cv2.cvtColor(
        image_np,
        cv2.COLOR_RGB2HSV
    )
)


h_channel = hsv[
    :,
    :,
    0
]

s_channel = hsv[
    :,
    :,
    1
]

v_channel = hsv[
    :,
    :,
    2
]


orange_support = (
    (
        h_channel
        >=
        3
    )
    &
    (
        h_channel
        <=
        27
    )
    &
    (
        s_channel
        >=
        90
    )
    &
    (
        v_channel
        >=
        65
    )
)


# ------------------------------------------------------------
# Evaluate each SAM proposal
# ------------------------------------------------------------

traffic_control_mask = np.zeros(
    (
        H,
        W
    ),
    dtype=bool
)


accepted_traffic_objects = []


for obj in traffic_objects:

    mask = obj[
        "mask"
    ]


    area = max(
        int(
            mask.sum()
        ),
        1
    )


    area_ratio = (
        area
        /
        (
            H
            *
            W
        )
    )


    ys, xs = np.where(
        mask
    )


    if (
        len(
            ys
        )
        ==
        0
    ):
        continue


    cy = (
        ys.mean()
        /
        H
    )


    # ----------------------------------------
    # Geometry filter
    # ----------------------------------------

    if (
        area_ratio
        >
        0.02
    ):
        continue


    if not (
        0.30
        <=
        cy
        <=
        0.92
    ):
        continue


    # ----------------------------------------
    # Street semantic support
    # ----------------------------------------

    surface_ratio = (
        street_surface_support[
            mask
        ]
        .mean()
    )


    if (
        surface_ratio
        <
        0.25
    ):
        continue


    # ----------------------------------------
    # Orange evidence
    # ----------------------------------------

    orange_ratio = (
        orange_support[
            mask
        ]
        .mean()
    )


    # ----------------------------------------
    # Mapillary evidence, when available
    # ----------------------------------------

    mapillary_ratio = (
        MAP_TRAFFIC_CONTROL[
            mask
        ]
        .mean()
    )


    # Require either orange evidence
    # OR direct Mapillary support.

    if (
        orange_ratio
        <
        0.025
        and
        mapillary_ratio
        <
        0.05
    ):
        continue


    gated_mask = (
        mask
        &
        street_surface_support
        &
        ~MAP_SKY
    )


    traffic_control_mask |= (
        gated_mask
    )


    accepted_traffic_objects.append({

        "score":
            obj["score"],

        "area_ratio":
            area_ratio,

        "surface_ratio":
            float(
                surface_ratio
            ),

        "orange_ratio":
            float(
                orange_ratio
            ),

        "mapillary_ratio":
            float(
                mapillary_ratio
            )
    })


# ------------------------------------------------------------
# Also accept direct Mapillary traffic-control pixels
# if such a class exists
# ------------------------------------------------------------

traffic_control_mask |= (

    MAP_TRAFFIC_CONTROL

    &
    street_surface_support
)


# ------------------------------------------------------------
# Don't overwrite person / vehicle
# ------------------------------------------------------------

protected_dynamic = np.isin(
    final_working,
    [
        LABEL2ID[
            "person"
        ],
        LABEL2ID[
            "vehicle"
        ]
    ]
)


traffic_control_mask &= (
    ~protected_dynamic
)


# ------------------------------------------------------------
# Install class
# ------------------------------------------------------------

final_working[
    traffic_control_mask
] = TRAFFIC_CONTROL_ID


print(
    "Accepted traffic-control objects:",
    len(
        accepted_traffic_objects
    )
)

print(
    "Traffic cone / barrel pixels:",
    int(
        traffic_control_mask.sum()
    )
)

print(
    "Traffic cone / barrel image %:",
    traffic_control_mask.mean()
    *
    100
)


if (
    len(
        accepted_traffic_objects
    )
    >
    0
):

    display(
        pd.DataFrame(
            accepted_traffic_objects
        )
    )


print(
    "✓ Traffic-control class installed"
)


In [ ]:
# ============================================================
# GOOGLE STREET VIEW ARTIFACT DETECTION — v1.5 STRICT IGNORE
#
# SCIENTIFIC RULE:
#   Google Street View screenshot UI / watermark residuals are
#   INVALID EVIDENCE, not scene objects.
#
# Therefore:
# - detect the residual footprint
# - set it to IGNORE=255
# - protect it permanently
# - NEVER assign a taxonomy class back into these pixels
# ============================================================

IGNORE_ID = IGNORE_INDEX if "IGNORE_INDEX" in globals() else 255

image_np = np.array(image)
gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)

# ------------------------------------------------------------
# 1. Search lower portion of Street View screenshot.
#    Dataset artifacts are dark, vertically persistent UI blocks
#    that touch the lower image boundary.
# ------------------------------------------------------------

bottom_search_region = np.zeros((H, W), dtype=bool)
bottom_search_region[int(H * 0.58):, :] = True

very_dark = gray < 58
artifact_dark = gray < 70
candidate_dark = artifact_dark & bottom_search_region

# Close Google lettering / striped internal gaps.
artifact_close_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (9, 21)
)

candidate_closed = cv2.morphologyEx(
    candidate_dark.astype(np.uint8),
    cv2.MORPH_CLOSE,
    artifact_close_kernel,
    iterations=2
) > 0

# ------------------------------------------------------------
# 2. Connected-component geometry gate
# ------------------------------------------------------------

num_components, artifact_labels, artifact_stats, artifact_centroids = (
    cv2.connectedComponentsWithStats(
        candidate_closed.astype(np.uint8),
        connectivity=8
    )
)

google_artifact_mask = np.zeros((H, W), dtype=bool)
accepted_artifact_components = []

for component_id in range(1, num_components):

    x = int(artifact_stats[component_id, cv2.CC_STAT_LEFT])
    y = int(artifact_stats[component_id, cv2.CC_STAT_TOP])
    w = int(artifact_stats[component_id, cv2.CC_STAT_WIDTH])
    h = int(artifact_stats[component_id, cv2.CC_STAT_HEIGHT])
    area = int(artifact_stats[component_id, cv2.CC_STAT_AREA])

    component_mask = artifact_labels == component_id

    bottom_touch = (y + h) >= (H - 3)
    width_ratio = w / W
    height_ratio = h / H
    area_ratio = area / (H * W)
    fill_ratio = area / max(w * h, 1)
    very_dark_ratio = float(very_dark[component_mask].mean())

    accept = (
        bottom_touch
        and y >= H * 0.55
        and 0.006 <= width_ratio <= 0.18
        and height_ratio >= 0.06
        and area_ratio <= 0.040
        and fill_ratio >= 0.24
        and very_dark_ratio >= 0.35
    )

    if not accept:
        continue

    # --------------------------------------------------------
    # Protect the WHOLE UI footprint, not only dark pixels.
    #
    # v1.2 intentionally uses a slightly more conservative
    # margin than v1.1 to catch antialiased edges and the pale
    # / gray remnants that could otherwise be called "pole".
    # --------------------------------------------------------

    pad_x = max(10, int(round(W * 0.0070)))
    pad_top = max(18, int(round(H * 0.0200)))

    x0 = max(0, x - pad_x)
    x1 = min(W, x + w + pad_x)
    y0 = max(0, y - pad_top)
    y1 = H

    component_bbox_mask = np.zeros((H, W), dtype=bool)
    component_bbox_mask[y0:y1, x0:x1] = True

    google_artifact_mask |= component_bbox_mask

    accepted_artifact_components.append({
        "component": int(component_id),
        "x": x,
        "y": y,
        "width": w,
        "height": h,
        "protected_x0": x0,
        "protected_y0": y0,
        "protected_x1": x1,
        "protected_y1": y1,
        "dark_component_pixels": area,
        "protected_pixels": int(component_bbox_mask.sum()),
        "image_pct": float(component_bbox_mask.mean() * 100.0),
        "very_dark_ratio": very_dark_ratio
    })

# Small final safety expansion for antialiased borders.
artifact_expand_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 7)
)

google_artifact_mask = cv2.dilate(
    google_artifact_mask.astype(np.uint8),
    artifact_expand_kernel,
    iterations=1
) > 0

artifact_ignore_mask = google_artifact_mask.copy()

# ============================================================
# v1.5.4 BORDER SEMANTIC ANOMALY / FRINGE DETECTOR
#
# Target failure mode:
#
# A narrow lower-left / lower-right Google smear fringe can be
# missed by image-only artifact detection and survive as:
#   upper_building_facade
#   ground_floor_solid_facade
#   other_unknown
#   pole_fixture
#
# We do NOT blindly mask border pixels.
#
# A component is accepted only when:
# - it lies in the lower border zone;
# - it has one of the suspicious semantic labels above;
# - it touches bottom or side edge;
# - nearby STREET-SURFACE context is sufficiently roadway-dominant.
# ============================================================

border_semantic_artifact_mask = np.zeros(
    (H, W),
    dtype=bool
)

border_semantic_rows = []

# ------------------------------------------------------------
# Search zone:
# lower 38% of image, only extreme left/right 9% of width.
# ------------------------------------------------------------

border_zone = np.zeros(
    (H, W),
    dtype=bool
)

border_y0 = int(
    H * 0.62
)

border_w = max(
    48,
    int(round(W * 0.09))
)

border_zone[
    border_y0:H,
    0:border_w
] = True

border_zone[
    border_y0:H,
    max(0, W - border_w):W
] = True


# ------------------------------------------------------------
# Semantic labels that are suspicious specifically at a
# bottom street-view border.
# ------------------------------------------------------------

border_suspicious_ids = [
    LABEL2ID[name]
    for name in [
        "upper_building_facade",
        "ground_floor_solid_facade",
        "other_unknown",
        "pole_fixture",
    ]
    if name in LABEL2ID
]

border_street_ids = [
    LABEL2ID[name]
    for name in [
        "roadway",
        "sidewalk",
        "bike_lane",
        "curb_edge",
    ]
    if name in LABEL2ID
]

ROAD_ID_BORDER = LABEL2ID[
    "roadway"
]

SIDEWALK_ID_BORDER = LABEL2ID[
    "sidewalk"
]

CURB_ID_BORDER = LABEL2ID[
    "curb_edge"
]


border_semantic_candidate = (
    border_zone
    &
    np.isin(
        final_working,
        border_suspicious_ids
    )
    &
    ~artifact_ignore_mask
)


# Join tiny gaps inside a vertical smear fringe.
border_semantic_candidate = cv2.morphologyEx(
    border_semantic_candidate.astype(np.uint8),
    cv2.MORPH_CLOSE,
    cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (5, 17)
    ),
    iterations=1
) > 0


n_bs, bs_lab, bs_stats, _ = cv2.connectedComponentsWithStats(
    border_semantic_candidate.astype(np.uint8),
    connectivity=8
)


for cid in range(
    1,
    n_bs
):

    comp = (
        bs_lab == cid
    )

    area = int(
        comp.sum()
    )

    if area == 0:
        continue

    x = int(
        bs_stats[
            cid,
            cv2.CC_STAT_LEFT
        ]
    )

    y = int(
        bs_stats[
            cid,
            cv2.CC_STAT_TOP
        ]
    )

    w = int(
        bs_stats[
            cid,
            cv2.CC_STAT_WIDTH
        ]
    )

    h = int(
        bs_stats[
            cid,
            cv2.CC_STAT_HEIGHT
        ]
    )

    touches_bottom = (
        y + h
        >=
        H - 2
    )

    touches_left = (
        x <= 2
    )

    touches_right = (
        x + w
        >=
        W - 2
    )

    touches_border = (
        touches_bottom
        or
        touches_left
        or
        touches_right
    )

    if not touches_border:
        continue

    area_ratio = (
        area
        /
        (H * W)
    )

    width_ratio = (
        w
        /
        W
    )

    height_ratio = (
        h
        /
        H
    )

    # --------------------------------------------------------
    # Local context ring around suspicious component.
    # --------------------------------------------------------

    margin_x = max(
        35,
        int(round(W * 0.035))
    )

    margin_y = max(
        25,
        int(round(H * 0.035))
    )

    x0 = max(
        0,
        x - margin_x
    )

    x1 = min(
        W,
        x + w + margin_x
    )

    y0 = max(
        border_y0,
        y - margin_y
    )

    y1 = min(
        H,
        y + h + margin_y
    )

    local_labels = final_working[
        y0:y1,
        x0:x1
    ]

    local_artifact = artifact_ignore_mask[
        y0:y1,
        x0:x1
    ]

    local_component = comp[
        y0:y1,
        x0:x1
    ]

    local_valid_street = (
        ~local_artifact
        &
        ~local_component
        &
        np.isin(
            local_labels,
            border_street_ids
        )
    )

    street_labels = local_labels[
        local_valid_street
    ]

    road_votes = int(
        np.sum(
            street_labels
            ==
            ROAD_ID_BORDER
        )
    )

    sidewalk_votes = int(
        np.sum(
            street_labels
            ==
            SIDEWALK_ID_BORDER
        )
    )

    curb_votes = int(
        np.sum(
            street_labels
            ==
            CURB_ID_BORDER
        )
    )

    street_votes = int(
        len(
            street_labels
        )
    )

    road_share = (
        road_votes
        /
        street_votes
        if street_votes > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Conservative acceptance:
    #
    # - small/narrow border anomaly
    # - tall enough to look like a smear fringe
    # - enough local street evidence
    # - roadway dominates context
    # --------------------------------------------------------

    narrow_enough = (
        width_ratio <= 0.085
    )

    tall_enough = (
        height_ratio >= 0.025
    )

    not_huge = (
        area_ratio <= 0.030
    )

    roadway_context = (
        street_votes >= 30
        and
        road_share >= 0.52
        and
        road_votes
        >=
        (
            sidewalk_votes
            +
            curb_votes
        )
    )

    accept = (
        touches_border
        and
        narrow_enough
        and
        tall_enough
        and
        not_huge
        and
        roadway_context
    )

    if accept:

        # -----------------------------------------------
        # Complete the suspicious fringe into a compact
        # rectangular bottom footprint.
        # -----------------------------------------------

        pad_x = max(
            6,
            int(round(W * 0.004))
        )

        pad_top = max(
            8,
            int(round(H * 0.012))
        )

        fx0 = max(
            0,
            x - pad_x
        )

        fx1 = min(
            W,
            x + w + pad_x
        )

        fy0 = max(
            border_y0,
            y - pad_top
        )

        fy1 = H

        border_semantic_artifact_mask[
            fy0:fy1,
            fx0:fx1
        ] = True

    border_semantic_rows.append({
        "component": cid,
        "x": x,
        "y": y,
        "width": w,
        "height": h,
        "pixels": area,
        "width_ratio": width_ratio,
        "height_ratio": height_ratio,
        "road_votes": road_votes,
        "sidewalk_votes": sidewalk_votes,
        "curb_votes": curb_votes,
        "street_votes": street_votes,
        "road_share": road_share,
        "accepted_as_artifact_fringe": accept,
    })


# Union semantic fringe with image-based artifact mask.
artifact_ignore_mask |= (
    border_semantic_artifact_mask
)

print(
    "Border semantic artifact fringe pixels:",
    int(
        border_semantic_artifact_mask.sum()
    )
)

if border_semantic_rows:

    display(
        pd.DataFrame(
            border_semantic_rows
        )
    )


# ============================================================

# Persist ONLY the successful border-fringe capture into the
# canonical Google artifact mask. No later footprint expansion.
google_artifact_mask = artifact_ignore_mask.copy()

# ------------------------------------------------------------
# 3. Install PERMANENT IGNORE
# ------------------------------------------------------------

artifact_pre_ignore_labels = final_working.copy()
final_working[artifact_ignore_mask] = IGNORE_ID

print("Detected Google/UI artifact components:", len(accepted_artifact_components))
print("STRICT IGNORE pixels:", int(artifact_ignore_mask.sum()))
print("STRICT IGNORE image %:", float(artifact_ignore_mask.mean() * 100.0))

if accepted_artifact_components:
    display(pd.DataFrame(accepted_artifact_components))

# ------------------------------------------------------------
# 4. Visual QA
# ------------------------------------------------------------

artifact_preview = image_np.copy()
artifact_preview[artifact_ignore_mask] = (255, 0, 255)

# Cyan = the v1.5.4 border semantic fringe capture only.
artifact_preview[
    border_semantic_artifact_mask
] = (
    0,
    255,
    255
)

plt.figure(figsize=(18, 8))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(artifact_preview)
plt.title("Magenta = original IGNORE; Cyan = v1.5.4 border fringe")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Google/UI artifact is permanently IGNORE=255")
print("✓ These pixels must never contribute to pole / road / sidewalk / any taxonomy class")


In [ ]:
# ============================================================
# COMMIT PRE-REFINEMENT RESULT — v1.4
#
# Compatibility variable name final_v010 is retained because
# later stable cells use it, but its contents are now:
#
#   v0.9 semantic backbone
#   + traffic-cone refinement
#   + Google artifact IGNORE
#
# It does NOT contain the legacy v0.10 tree overwrite.
# ============================================================

final_v010 = (
    final_working.copy()
)

print(
    "✓ v1.4 pre-refinement state committed"
)


In [ ]:
# ============================================================
# SCAFFOLD PRECISION FIX — v1.3 GLASS CURTAIN-WALL VETO
#
# Goal:
# - retain real temporary sidewalk shed / scaffold
# - reject glass-curtain-wall / glazed-façade false positives
# - restore rejected glass-like pixels to glazing, not UNKNOWN
# ============================================================

final_working = final_v010.copy()

SCAFFOLD_ID = LABEL2ID["sidewalk_shed_scaffold"]
ROAD_ID = LABEL2ID["roadway"]
SIDEWALK_ID = LABEL2ID["sidewalk"]
CURB_ID = LABEL2ID["curb_edge"]
UPPER_FACADE_ID = LABEL2ID["upper_building_facade"]
UPPER_GLASS_ID = LABEL2ID["upper_building_glazing"]
GROUND_GLASS_ID = LABEL2ID["ground_floor_glazing"]

current_scaffold = final_working == SCAFFOLD_ID
print("Scaffold before:", int(current_scaffold.sum()), "pixels")

building_support = MAP_BUILDING | ADE_BUILDING_V2
building_near = cv2.dilate(
    building_support.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (61, 41)),
    iterations=1
) > 0

# Strong glass evidence from three independent paths.
glazing_support = (
    (ADE_GLASS_V2 | MAP_WINDOW | precise_window_mask | facade_glass_aggressive)
    & building_support
)

# Stronger DINO scaffold evidence than the original fusion threshold.
strong_scaffold_dino = np.zeros((H, W), dtype=bool)
for obj in detail_objects:
    if (
        obj["class_name"] == "sidewalk_shed_scaffold"
        and obj["score"] >= 0.32
    ):
        strong_scaffold_dino |= obj["mask"]

num_labels, labels_scaf, stats_scaf, _ = cv2.connectedComponentsWithStats(
    current_scaffold.astype(np.uint8), connectivity=8
)

accepted_scaffold = np.zeros((H, W), dtype=bool)
scaffold_diagnostics = []

for cid in range(1, num_labels):
    component = labels_scaf == cid
    area = int(component.sum())
    if area == 0:
        continue

    x = int(stats_scaf[cid, cv2.CC_STAT_LEFT])
    y = int(stats_scaf[cid, cv2.CC_STAT_TOP])
    w = int(stats_scaf[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_scaf[cid, cv2.CC_STAT_HEIGHT])

    area_ratio = area / (H * W)
    building_ratio = float(building_near[component].mean())
    sidewalk_ratio = float(MAP_SIDEWALK[component].mean())
    glazing_ratio = float(glazing_support[component].mean())
    strong_dino_ratio = float(strong_scaffold_dino[component].mean())
    cy = (y + h / 2) / H
    vertical_ratio = h / max(w, 1)

    # Typical curtain-wall false positive:
    # vertically tall + glass-dominant + little sidewalk evidence.
    glass_tower_like = (
        glazing_ratio >= 0.22
        and vertical_ratio >= 1.10
        and sidewalk_ratio < 0.08
    )

    glass_panel_like = (
        glazing_ratio >= 0.42
        and strong_dino_ratio < 0.18
    )

    scaffold_evidence = (
        strong_dino_ratio >= 0.10
        or sidewalk_ratio >= 0.04
    )

    accept = (
        building_ratio >= 0.22
        and area_ratio <= 0.08
        and 0.20 <= cy <= 0.88
        and scaffold_evidence
        and not glass_tower_like
        and not glass_panel_like
    )

    if accept:
        accepted_scaffold[component] = True

    scaffold_diagnostics.append({
        "component": cid,
        "pixels": area,
        "image_pct": area_ratio * 100,
        "building_support": building_ratio,
        "sidewalk_support": sidewalk_ratio,
        "glazing_support": glazing_ratio,
        "strong_scaffold_dino": strong_dino_ratio,
        "vertical_ratio": vertical_ratio,
        "center_y": cy,
        "glass_tower_veto": glass_tower_like,
        "glass_panel_veto": glass_panel_like,
        "accepted": accept,
    })

rejected_scaffold = current_scaffold & ~accepted_scaffold
print("Accepted scaffold:", int(accepted_scaffold.sum()))
print("Rejected scaffold:", int(rejected_scaffold.sum()))
if scaffold_diagnostics:
    display(pd.DataFrame(scaffold_diagnostics))

# Reset rejected scaffold first.
final_working[rejected_scaffold] = UNKNOWN_ID

# Restore reliable street surfaces.
final_working[rejected_scaffold & MAP_ROAD] = ROAD_ID
final_working[rejected_scaffold & MAP_SIDEWALK] = SIDEWALK_ID
final_working[rejected_scaffold & MAP_CURB] = CURB_ID

# Restore building surface. Glazing gets priority over opaque façade.
rejected_building = rejected_scaffold & building_support
final_working[rejected_building] = UPPER_FACADE_ID

rejected_glass = rejected_scaffold & glazing_support

ground_glass_recovery = rejected_glass & interface_neighborhood
upper_glass_recovery = rejected_glass & ~interface_neighborhood

final_working[ground_glass_recovery] = GROUND_GLASS_ID
final_working[upper_glass_recovery] = UPPER_GLASS_ID

# Keep only scaffold components that survive the glass veto.
final_working[accepted_scaffold] = SCAFFOLD_ID

print("Recovered rejected scaffold as upper glazing:", int(upper_glass_recovery.sum()))
print("Recovered rejected scaffold as ground glazing:", int(ground_glass_recovery.sum()))
print("✓ v1.3 scaffold glass-curtain-wall safeguard applied")


In [ ]:
# ============================================================
# FENCE / RAILING RECOVERY — v1.3
#
# Purpose:
# Recover park perimeter fences / railings that remain
# other_unknown, without performing generic unknown filling.
# ============================================================

FENCE_ID = LABEL2ID["fence_railing"]

# 1. Semantic-model evidence
fence_semantic = MAP_FENCE | ADE_FENCE_V3

# 2. DINO + SAM fence evidence
fence_dino = np.zeros((H, W), dtype=bool)
for obj in detail_objects:
    if (
        obj["class_name"] == "fence_railing"
        and obj["score"] >= 0.18
    ):
        fence_dino |= obj["mask"]

# 3. Park/street-edge context: fence often runs next to sidewalk.
sidewalk_near_fence = cv2.dilate(
    MAP_SIDEWALK.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (101, 61)),
    iterations=1
) > 0

fence_candidate = (
    fence_semantic
    |
    (fence_dino & sidewalk_near_fence)
)

# Fence cannot be road, sky, glass façade, or Google UI artifact.
fence_candidate &= ~MAP_ROAD
fence_candidate &= ~MAP_SKY
fence_candidate &= ~ADE_GLASS_V2
fence_candidate &= ~MAP_WINDOW

if "artifact_ignore_mask" in globals():
    fence_candidate &= ~artifact_ignore_mask

# Preserve already reliable foreground/object classes.
protected_fence_classes = np.isin(
    final_working,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"],
        LABEL2ID["traffic_cone_barrel"],
        LABEL2ID["traffic_sign_signal"],
        LABEL2ID["sidewalk_shed_scaffold"],
        IGNORE_ID,
    ]
)
fence_candidate &= ~protected_fence_classes

# Deliberately conservative: v1.3 recovery only converts pixels
# that are still unknown (plus any already-fence pixels).
fence_recovery = (
    fence_candidate
    & np.isin(final_working, [UNKNOWN_ID, FENCE_ID])
)

# Remove tiny isolated speckle while retaining thin openwork fence lines.
num_f, lab_f, stats_f, _ = cv2.connectedComponentsWithStats(
    fence_recovery.astype(np.uint8), connectivity=8
)
fence_precise = np.zeros((H, W), dtype=bool)
fence_rows = []

for cid in range(1, num_f):
    comp = lab_f == cid
    area = int(comp.sum())
    x = int(stats_f[cid, cv2.CC_STAT_LEFT])
    y = int(stats_f[cid, cv2.CC_STAT_TOP])
    w = int(stats_f[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_f[cid, cv2.CC_STAT_HEIGHT])
    cy = (y + h/2) / H
    area_ratio = area / (H*W)
    semantic_ratio = float(fence_semantic[comp].mean()) if area else 0.0
    dino_ratio = float(fence_dino[comp].mean()) if area else 0.0

    accept = (
        area >= 8
        and area_ratio <= 0.035
        and 0.18 <= cy <= 0.90
        and (semantic_ratio >= 0.05 or dino_ratio >= 0.05)
    )

    if accept:
        fence_precise[comp] = True

    fence_rows.append({
        "component": cid,
        "pixels": area,
        "width": w,
        "height": h,
        "center_y": cy,
        "semantic_support": semantic_ratio,
        "dino_support": dino_ratio,
        "accepted": accept,
    })

unknown_before_fence = int(np.sum(final_working == UNKNOWN_ID))
final_working[fence_precise] = FENCE_ID
unknown_after_fence = int(np.sum(final_working == UNKNOWN_ID))

print("Fence semantic pixels:", int(fence_semantic.sum()))
print("Fence DINO/SAM pixels:", int(fence_dino.sum()))
print("Recovered fence pixels:", int(fence_precise.sum()))
print("Unknown reduction from fence recovery:", unknown_before_fence - unknown_after_fence)
if fence_rows:
    display(pd.DataFrame(fence_rows))
print("✓ v1.3 fence / railing recovery applied")


In [ ]:
# ============================================================
# v1.5 LOCAL RECOVERY
#
# A. entrance awning / canopy
# B. solid boundary wall → wall_ledge
# C. precise woody trunk / bare branch → tree
#
# No broad foliage write is permitted.
# ============================================================

AWNING_ID = LABEL2ID["awning_canopy"]
WALL_ID = LABEL2ID["wall_ledge"]
TREE_ID = LABEL2ID["tree"]

# ------------------------------------------------------------
# Shared semantic helpers
# ------------------------------------------------------------

ADE_WALL_V15 = ade_mask_fuzzy(
    ["wall"],
    ["building"]
)

MAP_WALL_V15 = mapillary_mask(
    ["wall"]
)

MAP_POLE_V15 = mapillary_mask(
    ["pole"]
)

MAP_VEGETATION_V15 = mapillary_mask(
    ["vegetation", "tree", "plant"]
)

image_rgb_v15 = np.array(image)
gray_v15 = cv2.cvtColor(
    image_rgb_v15,
    cv2.COLOR_RGB2GRAY
)

# ============================================================
# v1.5.6 SCAFFOLD PROTECTED ZONE
#
# Purpose:
# Prevent a real sidewalk shed / scaffold from being fragmented
# into awning_canopy + wall_ledge by later local recovery.
#
# This is intentionally LOCAL:
#
# accepted_scaffold
#   +
# strong_scaffold_dino only where it remains close to an
# already accepted scaffold component.
#
# It does NOT expand scaffold across the whole frontage.
# ============================================================

SCAFFOLD_LOCAL_ID = LABEL2ID[
    "sidewalk_shed_scaffold"
]

if (
    "accepted_scaffold" in globals()
    and
    "strong_scaffold_dino" in globals()
):

    scaffold_anchor_near = cv2.dilate(
        accepted_scaffold.astype(np.uint8),
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (35, 25)
        ),
        iterations=1
    ) > 0

    scaffold_protected_zone = (
        accepted_scaffold
        |
        (
            strong_scaffold_dino
            &
            scaffold_anchor_near
        )
    )

else:

    scaffold_protected_zone = np.zeros(
        (H, W),
        dtype=bool
    )


# Protect Google artifact / dynamic objects from this guard.
if "artifact_ignore_mask" in globals():
    scaffold_protected_zone &= ~artifact_ignore_mask

scaffold_protected_zone &= ~np.isin(
    final_working,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"],
        LABEL2ID["traffic_cone_barrel"],
        LABEL2ID["traffic_sign_signal"],
        IGNORE_ID,
    ]
)


# Reclaim any pre-existing awning / wall label that sits inside
# evidence-backed scaffold structure.
preexisting_scaffold_conflict = (
    scaffold_protected_zone
    &
    np.isin(
        final_working,
        [
            AWNING_ID,
            WALL_ID,
        ]
    )
)

final_working[
    preexisting_scaffold_conflict
] = SCAFFOLD_LOCAL_ID


print("========================================")
print("SCAFFOLD PROTECTED ZONE — v1.5.6")
print("========================================")
print(
    "Protected scaffold-context pixels:",
    int(scaffold_protected_zone.sum())
)
print(
    "Pre-existing awning/wall pixels reclaimed:",
    int(preexisting_scaffold_conflict.sum())
)


# ============================================================
# A. ENTRANCE AWNING RECOVERY
# ============================================================

awning_dino = np.zeros((H, W), dtype=bool)
awning_high = np.zeros((H, W), dtype=bool)

for obj in detail_objects:
    if obj["class_name"] != "awning_canopy":
        continue

    if obj["score"] >= 0.18:
        awning_dino |= obj["mask"]

    if obj["score"] >= 0.24:
        awning_high |= obj["mask"]

building_near_awning = cv2.dilate(
    building_context.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (51, 41)),
    iterations=1
) > 0

interface_near_awning = cv2.dilate(
    interface_neighborhood.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (31, 25)),
    iterations=1
) > 0

awning_candidate = (
    awning_dino
    &
    building_near_awning
    &
    interface_near_awning
)

# Do not overwrite street surfaces or research foreground objects.
awning_candidate &= ~MAP_ROAD
awning_candidate &= ~MAP_SIDEWALK
awning_candidate &= ~MAP_SKY

# v1.5.6: scaffold evidence has precedence over awning recovery.
awning_candidate &= ~scaffold_protected_zone

awning_candidate &= ~np.isin(
    final_working,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"],
        LABEL2ID["traffic_cone_barrel"],
        LABEL2ID["traffic_sign_signal"],
        LABEL2ID["sidewalk_shed_scaffold"],
        IGNORE_ID,
    ]
)

if "artifact_ignore_mask" in globals():
    awning_candidate &= ~artifact_ignore_mask

num_a, lab_a, stats_a, _ = cv2.connectedComponentsWithStats(
    awning_candidate.astype(np.uint8),
    connectivity=8
)

awning_precise = np.zeros((H, W), dtype=bool)
awning_rows = []

for cid in range(1, num_a):
    comp = lab_a == cid
    area = int(comp.sum())
    if area == 0:
        continue

    x = int(stats_a[cid, cv2.CC_STAT_LEFT])
    y = int(stats_a[cid, cv2.CC_STAT_TOP])
    w = int(stats_a[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_a[cid, cv2.CC_STAT_HEIGHT])

    area_ratio = area / (H * W)
    aspect = w / max(h, 1)
    cy = (y + h / 2) / H
    building_ratio = float(building_near_awning[comp].mean())
    interface_ratio = float(interface_near_awning[comp].mean())
    high_ratio = float(awning_high[comp].mean())

    keep = (
        area >= 20
        and area_ratio <= 0.030
        and 0.22 <= cy <= 0.82
        and aspect >= 0.90
        and building_ratio >= 0.45
        and interface_ratio >= 0.15
        and high_ratio >= 0.02
    )

    if keep:
        awning_precise[comp] = True

    awning_rows.append({
        "component": cid,
        "pixels": area,
        "aspect_w_h": aspect,
        "center_y": cy,
        "building_support": building_ratio,
        "interface_support": interface_ratio,
        "high_conf_support": high_ratio,
        "accepted": keep,
    })

final_working[awning_precise] = AWNING_ID

print("========================================")
print("AWNING RECOVERY")
print("========================================")
print("Recovered awning pixels:", int(awning_precise.sum()))
if awning_rows:
    display(pd.DataFrame(awning_rows))


# ============================================================
# B. SOLID BOUNDARY WALL RECOVERY
#
# The taxonomy already contains wall_ledge for low / raised
# solid walls.  Recover only pixels still marked other_unknown.
# ============================================================

wall_dino = np.zeros((H, W), dtype=bool)
for obj in detail_objects:
    if (
        obj["class_name"] == "wall_ledge"
        and obj["score"] >= 0.18
    ):
        wall_dino |= obj["mask"]

wall_semantic = (
    ADE_WALL_V15
    |
    MAP_WALL_V15
)

wall_sidewalk_near = cv2.dilate(
    MAP_SIDEWALK.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (111, 71)),
    iterations=1
) > 0

wall_candidate = (
    (wall_semantic | wall_dino)
    &
    wall_sidewalk_near
    &
    (final_working == UNKNOWN_ID)
)

wall_candidate &= ~MAP_ROAD
wall_candidate &= ~MAP_SKY
wall_candidate &= ~MAP_WINDOW
wall_candidate &= ~ADE_GLASS_V2
wall_candidate &= ~(final_working == TREE_ID)

# v1.5.6: evidence-backed scaffold zone has precedence over
# wall_ledge recovery.
wall_candidate &= ~scaffold_protected_zone

if "artifact_ignore_mask" in globals():
    wall_candidate &= ~artifact_ignore_mask

num_w, lab_w, stats_w, _ = cv2.connectedComponentsWithStats(
    wall_candidate.astype(np.uint8),
    connectivity=8
)

wall_precise = np.zeros((H, W), dtype=bool)
wall_rows = []

for cid in range(1, num_w):
    comp = lab_w == cid
    area = int(comp.sum())
    if area == 0:
        continue

    x = int(stats_w[cid, cv2.CC_STAT_LEFT])
    y = int(stats_w[cid, cv2.CC_STAT_TOP])
    w = int(stats_w[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_w[cid, cv2.CC_STAT_HEIGHT])

    area_ratio = area / (H * W)
    cy = (y + h / 2) / H
    semantic_ratio = float(wall_semantic[comp].mean())
    dino_ratio = float(wall_dino[comp].mean())

    keep = (
        area >= 20
        and area_ratio <= 0.090
        and 0.18 <= cy <= 0.92
        and (
            semantic_ratio >= 0.05
            or dino_ratio >= 0.04
        )
    )

    if keep:
        wall_precise[comp] = True

    wall_rows.append({
        "component": cid,
        "pixels": area,
        "center_y": cy,
        "semantic_support": semantic_ratio,
        "dino_support": dino_ratio,
        "accepted": keep,
    })

final_working[wall_precise] = WALL_ID

print()
print("========================================")
print("BOUNDARY WALL RECOVERY")
print("========================================")
print("Recovered wall_ledge pixels:", int(wall_precise.sum()))
if wall_rows:
    display(pd.DataFrame(wall_rows))


# ============================================================
# C. PRECISE WOODY TREE RECOVERY
#
# v1.4 prohibited all DINO/SAM tree writes because broad SAM
# foliage contaminated glazing.
#
# v1.5 restores ONLY trunk / bare-branch structure.
#
# Requirements:
# - inside a woody DINO/SAM proposal
# - dark and/or edge-like image structure
# - near an existing semantic tree anchor OR high-confidence
#   woody proposal with strong elongated/sparse geometry
# - never Google UI / road / sidewalk / dynamic object
#
# Broad foliage remains diagnostic only.
# ============================================================

raw_woody_v15 = np.zeros((H, W), dtype=bool)
high_woody_v15 = np.zeros((H, W), dtype=bool)

for obj in tree_objects:
    if obj["component"] != "woody":
        continue

    if obj["score"] >= 0.16:
        raw_woody_v15 |= obj["mask"]

    if obj["score"] >= 0.22:
        high_woody_v15 |= obj["mask"]

tree_semantic_anchor = (
    backbone_tree_mask
    |
    ADE_TREE_V2
    |
    MAP_VEGETATION_V15
)

tree_anchor_near = cv2.dilate(
    tree_semantic_anchor.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (101, 101)),
    iterations=1
) > 0

tree_anchor_contact = cv2.dilate(
    tree_semantic_anchor.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31)),
    iterations=1
) > 0

edge_v15 = cv2.Canny(
    gray_v15,
    45,
    135
) > 0

edge_support_v15 = cv2.dilate(
    edge_v15.astype(np.uint8),
    cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),
    iterations=1
) > 0

dark_woody_v15 = gray_v15 < 165
very_dark_woody_v15 = gray_v15 < 105

woody_pixel_structure = (
    dark_woody_v15
    &
    (
        edge_support_v15
        |
        very_dark_woody_v15
    )
)

woody_candidate = (
    raw_woody_v15
    &
    woody_pixel_structure
    &
    tree_anchor_near
)

# Allowed occlusion surfaces: a real branch/trunk can be seen
# in front of sky or building/glazing.
woody_allowed_source = np.isin(
    final_working,
    [
        TREE_ID,
        LABEL2ID["sky"],
        LABEL2ID["upper_building_facade"],
        LABEL2ID["upper_building_glazing"],
        LABEL2ID["ground_floor_solid_facade"],
        LABEL2ID["ground_floor_glazing"],
        UNKNOWN_ID,
    ]
)

woody_candidate &= woody_allowed_source
woody_candidate &= ~MAP_ROAD
woody_candidate &= ~MAP_SIDEWALK
woody_candidate &= ~MAP_CURB
woody_candidate &= ~MAP_POLE_V15

woody_candidate &= ~np.isin(
    final_working,
    [
        LABEL2ID["person"],
        LABEL2ID["vehicle"],
        LABEL2ID["traffic_cone_barrel"],
        LABEL2ID["traffic_sign_signal"],
        LABEL2ID["signboard"],
        LABEL2ID["awning_canopy"],
        LABEL2ID["sidewalk_shed_scaffold"],
        LABEL2ID["fence_railing"],
        LABEL2ID["wall_ledge"],
        IGNORE_ID,
    ]
)

if "artifact_ignore_mask" in globals():
    woody_candidate &= ~artifact_ignore_mask

# Connect small gaps in branch/trunk lines without expanding into
# a broad silhouette.
woody_connected = cv2.morphologyEx(
    woody_candidate.astype(np.uint8),
    cv2.MORPH_CLOSE,
    cv2.getStructuringElement(cv2.MORPH_RECT, (3, 5)),
    iterations=1
) > 0

num_t, lab_t, stats_t, _ = cv2.connectedComponentsWithStats(
    woody_connected.astype(np.uint8),
    connectivity=8
)

woody_recovery_precise = np.zeros((H, W), dtype=bool)
woody_rows = []

for cid in range(1, num_t):
    comp = lab_t == cid
    area = int(comp.sum())
    if area == 0:
        continue

    x = int(stats_t[cid, cv2.CC_STAT_LEFT])
    y = int(stats_t[cid, cv2.CC_STAT_TOP])
    w = int(stats_t[cid, cv2.CC_STAT_WIDTH])
    h = int(stats_t[cid, cv2.CC_STAT_HEIGHT])

    area_ratio = area / (H * W)
    fill_ratio = area / max(w * h, 1)
    elongation = max(
        h / max(w, 1),
        w / max(h, 1)
    )
    vertical_span = h / H
    anchor_contact_ratio = float(tree_anchor_contact[comp].mean())
    high_conf_ratio = float(high_woody_v15[comp].mean())

    sparse_branch_network = fill_ratio <= 0.42
    elongated_structure = elongation >= 1.45

    independent_high_conf_trunk = (
        high_conf_ratio >= 0.20
        and vertical_span >= 0.07
        and elongation >= 1.35
    )

    keep = (
        area >= 8
        and area_ratio <= 0.014
        and (
            sparse_branch_network
            or elongated_structure
        )
        and (
            anchor_contact_ratio >= 0.01
            or independent_high_conf_trunk
        )
    )

    if keep:
        woody_recovery_precise[comp] = True

    woody_rows.append({
        "component": cid,
        "pixels": area,
        "width": w,
        "height": h,
        "fill_ratio": fill_ratio,
        "elongation": elongation,
        "vertical_span": vertical_span,
        "tree_anchor_contact": anchor_contact_ratio,
        "high_conf_woody": high_conf_ratio,
        "accepted": keep,
    })

# Install only precise woody structure.
tree_before_woody_v15 = int(np.sum(final_working == TREE_ID))
final_working[woody_recovery_precise] = TREE_ID
tree_after_woody_v15 = int(np.sum(final_working == TREE_ID))

print()
print("========================================")
print("WOODY TREE RECOVERY")
print("========================================")
print("Tree pixels before:", tree_before_woody_v15)
print("Recovered trunk / branch pixels:", int(woody_recovery_precise.sum()))
print("Tree pixels after:", tree_after_woody_v15)
if woody_rows:
    display(pd.DataFrame(woody_rows))

# ============================================================
# v1.5.6 FINAL SCAFFOLD PRECEDENCE RE-ASSERTION
#
# Defensive final guard:
# later local recovery may not leave awning_canopy / wall_ledge
# inside the evidence-backed scaffold zone.
# ============================================================

late_scaffold_conflict = (
    scaffold_protected_zone
    &
    np.isin(
        final_working,
        [
            AWNING_ID,
            WALL_ID,
        ]
    )
)

final_working[
    late_scaffold_conflict
] = SCAFFOLD_LOCAL_ID


remaining_scaffold_awning_wall_conflict = (
    scaffold_protected_zone
    &
    np.isin(
        final_working,
        [
            AWNING_ID,
            WALL_ID,
        ]
    )
)

assert not np.any(
    remaining_scaffold_awning_wall_conflict
), (
    "SCAFFOLD PRECEDENCE FAILURE: awning_canopy or wall_ledge "
    "remains inside evidence-backed scaffold context."
)


print()
print("========================================")
print("SCAFFOLD PRECEDENCE QA — v1.5.6")
print("========================================")
print(
    "Late awning/wall conflicts reclaimed:",
    int(late_scaffold_conflict.sum())
)
print(
    "Remaining scaffold-context awning/wall conflicts:",
    int(remaining_scaffold_awning_wall_conflict.sum())
)

print()
print("✓ v1.5.6 local recovery complete")


In [ ]:
# ============================================================
# TREE / GLAZING BACKBONE LOCK — v1.5
#
# No tree reconstruction is performed.
#
# This cell only enforces regression invariants:
#
# 1. Pixels that were glazing in the semantic backbone may not
#    become tree because of a later tree refinement.
#
# 2. Existing v0.9 tree pixels are retained unless a later
#    foreground-specific class legitimately occupies them.
#
# There is intentionally NO:
#
#   final_working[raw_foliage] = TREE_ID
#
# and NO:
#
#   final_working[raw_woody] = TREE_ID
# ============================================================

tree_on_backbone_glazing = (
    backbone_glazing_mask
    &
    (
        final_working
        ==
        TREE_ID
    )
)

# v1.5 permits only the explicitly approved woody branch/trunk
# recovery to cross a protected glazing surface.
illegal_tree_on_glazing = (
    tree_on_backbone_glazing
    &
    ~woody_recovery_precise
)

if np.any(
    illegal_tree_on_glazing
):
    final_working[
        illegal_tree_on_glazing
    ] = semantic_backbone[
        illegal_tree_on_glazing
    ]


print(
    "Approved woody tree pixels crossing protected glazing:",
    int(
        np.sum(
            backbone_glazing_mask
            &
            (
                final_working
                ==
                TREE_ID
            )
        )
    )
)


assert not np.any(
    backbone_glazing_mask
    &
    (
        final_working
        ==
        TREE_ID
    )
    &
    ~woody_recovery_precise
)


print(
    "✓ v1.5 tree/glazing regression guard passed"
)


In [ ]:
# ============================================================
# POLE + TRAFFIC SIGNAL RECOVERY
#
# pole_fixture:
# - streetlight pole
# - utility pole
# - traffic-signal support pole / mast
#
# traffic_sign_signal:
# - traffic light head
# - traffic sign
# ============================================================

POLE_ID = LABEL2ID[
    "pole_fixture"
]

TRAFFIC_SIGNAL_ID = LABEL2ID[
    "traffic_sign_signal"
]


# ============================================================
# MAPILLARY SUPPORT
# ============================================================

MAP_POLE = mapillary_mask(
    [
        "pole"
    ]
)


MAP_TRAFFIC_LIGHT = mapillary_mask(
    [
        "traffic light"
    ]
)


MAP_TRAFFIC_SIGN = mapillary_mask(
    [
        "traffic sign"
    ]
)


print(
    "Mapillary pole:",
    int(
        MAP_POLE.sum()
    )
)

print(
    "Mapillary traffic light:",
    int(
        MAP_TRAFFIC_LIGHT.sum()
    )
)

print(
    "Mapillary traffic sign:",
    int(
        MAP_TRAFFIC_SIGN.sum()
    )
)


# ============================================================
# ADE SUPPORT
# ============================================================

ADE_POLE_CURRENT = ade_mask_fuzzy(
    [
        "pole",
        "streetlight",
        "street light"
    ]
)


ADE_TRAFFIC_LIGHT_CURRENT = ade_mask_fuzzy(
    [
        "traffic light"
    ]
)


ADE_TRAFFIC_SIGN_CURRENT = ade_mask_fuzzy(
    [
        "traffic sign"
    ]
)


# ------------------------------------------------------------
# Combined proposal
# ------------------------------------------------------------

pole_candidate = (

    MAP_POLE

    |

    ADE_POLE_CURRENT
)


traffic_signal_candidate = (

    MAP_TRAFFIC_LIGHT

    |

    MAP_TRAFFIC_SIGN

    |

    ADE_TRAFFIC_LIGHT_CURRENT

    |

    ADE_TRAFFIC_SIGN_CURRENT
)


# ============================================================
# POLE GEOMETRY FILTER
#
# Pole should generally be a vertically persistent component.
# ============================================================

num_poles, pole_labels, pole_stats, _ = (
    cv2.connectedComponentsWithStats(
        pole_candidate.astype(np.uint8),
        connectivity=8
    )
)


pole_precise = np.zeros(
    (H, W),
    dtype=bool
)


pole_rows = []


for cid in range(
    1,
    num_poles
):

    component = (
        pole_labels
        ==
        cid
    )


    area = int(
        component.sum()
    )


    x = pole_stats[
        cid,
        cv2.CC_STAT_LEFT
    ]

    y = pole_stats[
        cid,
        cv2.CC_STAT_TOP
    ]

    w = pole_stats[
        cid,
        cv2.CC_STAT_WIDTH
    ]

    h = pole_stats[
        cid,
        cv2.CC_STAT_HEIGHT
    ]


    area_ratio = (
        area
        /
        (H * W)
    )


    vertical_ratio = (
        h
        /
        max(
            w,
            1
        )
    )


    # Allow:
    # - long vertical poles
    # - smaller distant poles

    accept = (

        area_ratio
        <=
        0.018

        and

        h
        >=
        H * 0.025

        and

        vertical_ratio
        >=
        1.35
    )


    if accept:

        pole_precise[
            component
        ] = True


    pole_rows.append({

        "component":
            cid,

        "pixels":
            area,

        "width":
            w,

        "height":
            h,

        "vertical_ratio":
            vertical_ratio,

        "accepted":
            accept
    })


if pole_rows:

    display(
        pd.DataFrame(
            pole_rows
        )
    )


# ------------------------------------------------------------
# Never call road / sidewalk pixels pole unless semantic model
# explicitly returned the pole there.
# ------------------------------------------------------------

pole_precise &= (
    ~MAP_SKY
)


# ------------------------------------------------------------
# Protect dynamic / research objects
# ------------------------------------------------------------

do_not_overwrite = np.isin(
    final_working,
    [

        LABEL2ID[
            "person"
        ],

        LABEL2ID[
            "vehicle"
        ],

        LABEL2ID[
            "traffic_cone_barrel"
        ],

        IGNORE_ID
    ]
)


pole_precise &= (
    ~do_not_overwrite
)


traffic_signal_candidate &= (
    ~do_not_overwrite
)

# Absolute protection: a detected Street View screenshot artifact
# must never be reclassified as a pole / sign / signal.
if "google_artifact_mask" in globals():
    pole_precise &= ~google_artifact_mask
    traffic_signal_candidate &= ~google_artifact_mask

# v1.5: a woody component that passed the strict trunk/branch
# recovery gate must not be stolen back by the pole recovery.
if "woody_recovery_precise" in globals():
    pole_precise &= ~woody_recovery_precise
    traffic_signal_candidate &= ~woody_recovery_precise


# ------------------------------------------------------------
# Install pole first
# ------------------------------------------------------------

final_working[
    pole_precise
] = POLE_ID


# Signal/sign head has priority over its pole

final_working[
    traffic_signal_candidate
] = TRAFFIC_SIGNAL_ID


print(
    "Final pole pixels:",
    int(
        pole_precise.sum()
    )
)

print(
    "Final traffic sign/signal pixels:",
    int(
        traffic_signal_candidate.sum()
    )
)

print(
    "✓ Pole / traffic signal recovery applied"
)


In [ ]:
# ============================================================
# v1.5 BACKBONE REGRESSION AUDIT
#
# The purpose is not to force later labels to equal v0.9.
# Foreground classes such as pole / traffic signal may
# legitimately occlude a background class.
#
# The audit specifically checks the known regressions:
# - glazing → tree
# - broad sidewalk → stoop
# - Google artifact → semantic object
# ============================================================

# 1. Glazing must never become tree.
illegal_glazing_to_tree_pixels = int(
    np.sum(
        backbone_glazing_mask
        &
        (
            final_working
            ==
            LABEL2ID[
                "tree"
            ]
        )
        &
        ~woody_recovery_precise
    )
)

approved_woody_on_glazing_pixels = int(
    np.sum(
        backbone_glazing_mask
        &
        (
            final_working
            ==
            LABEL2ID[
                "tree"
            ]
        )
        &
        woody_recovery_precise
    )
)


# 2. Any surviving stoop must belong to a component that already
#    passed the v1.4 component-level stoop precision gate.
#
# IMPORTANT:
# v1.4 validates stoop by connected component, not by requiring
# every pixel to sit inside entrance_neighborhood.
#
# Therefore the regression audit must compare against
# valid_stoop_v14, which is the authoritative approved-component
# mask created by the stoop precision stage.
final_stoop_mask = (
    final_working
    ==
    LABEL2ID[
        "stoop_stair"
    ]
)


unsupported_final_stoop = (
    final_stoop_mask
    &
    ~valid_stoop_v14
)


# 3. Google artifact remains IGNORE.
artifact_semantic_contamination = 0

if "artifact_ignore_mask" in globals():

    artifact_semantic_contamination = int(
        np.sum(
            artifact_ignore_mask
            &
            (
                final_working
                !=
                IGNORE_ID
            )
        )
    )


print(
    "========================================"
)

print(
    "v1.5 BACKBONE REGRESSION AUDIT"
)

print(
    "========================================"
)

print(
    "Illegal glazing → tree:",
    illegal_glazing_to_tree_pixels
)

print(
    "Approved woody crossings on glazing:",
    approved_woody_on_glazing_pixels
)

print(
    "Final stoop pixels outside approved components:",
    int(
        unsupported_final_stoop.sum()
    )
)

print(
    "Google artifact semantic contamination:",
    artifact_semantic_contamination
)


assert illegal_glazing_to_tree_pixels == 0

assert not np.any(
    unsupported_final_stoop
), (
    "Stoop regression detected: final output contains stoop pixels "
    "outside components approved by the v1.4 stoop precision gate."
)

assert artifact_semantic_contamination == 0


print(
    "✓ v1.5 regression audit passed"
)


In [ ]:
# ============================================================
# STREET VIEW ARTIFACT FINAL LOCK — v1.5
#
# IMPORTANT:
# Scientific data:
#   artifact pixels stay IGNORE=255 permanently.
#
# Display only:
#   create a separate contextual visualization fill so the
#   ignored screenshot block does not appear as a gray/black
#   object in the figure.
#
# The display fill is NEVER copied back into final_working.
# ============================================================

from scipy.ndimage import distance_transform_edt

if "google_artifact_mask" not in globals():
    google_artifact_mask = np.zeros((H, W), dtype=bool)

artifact_ignore_mask = google_artifact_mask.copy()
artifact_detected_pixels = int(artifact_ignore_mask.sum())

# ------------------------------------------------------------
# 1. HARD FINAL LOCK
#
# Even if an earlier/later refinement accidentally touched the
# area, this line wins immediately before final commit.
# ------------------------------------------------------------

final_working[artifact_ignore_mask] = IGNORE_ID

# Compatibility audit variables.
artifact_restored_mask = np.zeros((H, W), dtype=bool)
artifact_unresolved_mask = artifact_ignore_mask.copy()

# ------------------------------------------------------------
# 2. DISPLAY-ONLY contextual fill
#
# Use only broad scene-surface / vegetation classes as donors.
# Explicitly EXCLUDE narrow object classes that could recreate
# the screenshot shape:
#   pole, sign/signal, person, vehicle, cone, signboard, etc.
# ------------------------------------------------------------

DISPLAY_CONTEXT_CLASS_NAMES = [
    "roadway",
    "sidewalk",
    "bike_lane",
    "curb_edge",
    "upper_building_facade",
    "ground_floor_solid_facade",
    "ground_floor_glazing",
    "upper_building_glazing",
    "door_entrance",
    "awning_canopy",
    "sidewalk_shed_scaffold",
    "tree",
    "shrub_hedge",
    "ground_vegetation",
    "vertical_green_wall",
    "sky",
]

display_context_ids = [
    LABEL2ID[name]
    for name in DISPLAY_CONTEXT_CLASS_NAMES
    if name in LABEL2ID
]

display_context_labels = final_working.copy()

display_donor_mask = (
    ~artifact_ignore_mask
    &
    np.isin(final_working, display_context_ids)
    &
    (final_working != IGNORE_ID)
)

if artifact_detected_pixels > 0 and display_donor_mask.any():

    # --------------------------------------------------------
    # PASS 1 — row-aware street-surface restoration.
    #
    # Google screenshot residuals in these SVI exports are
    # vertical columns.  Filling from the same image row allows
    # the upper part of a residual to inherit sidewalk while the
    # lower part inherits roadway, instead of painting the whole
    # block as one class.
    # --------------------------------------------------------

    street_surface_ids = [
        LABEL2ID[name]
        for name in [
            "roadway",
            "sidewalk",
            "bike_lane",
            "curb_edge",
        ]
        if name in LABEL2ID
    ]

    display_filled = np.zeros((H, W), dtype=bool)

    artifact_rows = np.where(
        artifact_ignore_mask.any(axis=1)
    )[0]

    for yy in artifact_rows:
        target_x = np.where(
            artifact_ignore_mask[yy]
        )[0]

        donor_x = np.where(
            (~artifact_ignore_mask[yy])
            &
            np.isin(
                final_working[yy],
                street_surface_ids
            )
        )[0]

        if len(target_x) == 0 or len(donor_x) == 0:
            continue

        # Vectorized nearest horizontal donor.
        pos = np.searchsorted(
            donor_x,
            target_x
        )

        left_pos = np.clip(
            pos - 1,
            0,
            len(donor_x) - 1
        )

        right_pos = np.clip(
            pos,
            0,
            len(donor_x) - 1
        )

        left_x = donor_x[left_pos]
        right_x = donor_x[right_pos]

        choose_right = (
            np.abs(right_x - target_x)
            <
            np.abs(target_x - left_x)
        )

        nearest_x_row = np.where(
            choose_right,
            right_x,
            left_x
        )

        display_context_labels[
            yy,
            target_x
        ] = final_working[
            yy,
            nearest_x_row
        ]

        display_filled[
            yy,
            target_x
        ] = True

    # --------------------------------------------------------
    # PASS 2 — fallback for artifact pixels where no road /
    # sidewalk / curb donor exists in exactly the same row.
    #
    # Use a strong horizontal bias and only non-UNKNOWN broad
    # context classes.
    # --------------------------------------------------------

    unresolved_display = (
        artifact_ignore_mask
        &
        ~display_filled
    )

    if np.any(unresolved_display):

        search_space = ~display_donor_mask

        _, nearest_indices = distance_transform_edt(
            search_space,
            sampling=(6.0, 1.0),
            return_distances=True,
            return_indices=True
        )

        nearest_y = nearest_indices[0]
        nearest_x = nearest_indices[1]

        contextual_fill = final_working[
            nearest_y,
            nearest_x
        ]

        display_context_labels[
            unresolved_display
        ] = contextual_fill[
            unresolved_display
        ]

else:
    display_context_labels[artifact_ignore_mask] = UNKNOWN_ID

# Absolute display safety:
# even if future edits alter donor logic, Google/UI pixels may
# never display as pole_fixture or traffic_sign_signal.
for forbidden_name in [
    "pole_fixture",
    "traffic_sign_signal",
    "person",
    "vehicle",
    "traffic_cone_barrel",
    "signboard",
]:
    if forbidden_name in LABEL2ID:
        bad_display = (
            artifact_ignore_mask
            &
            (display_context_labels == LABEL2ID[forbidden_name])
        )
        display_context_labels[bad_display] = UNKNOWN_ID

# ============================================================
# v1.5.6 BOTTOM ARTIFACT ROADWAY-CONTEXT DISPLAY LOCK
#
# IMPORTANT:
# This operates ONLY on pixels already confirmed by
# artifact_ignore_mask.
#
# It does NOT:
# - expand the artifact mask;
# - modify normal sidewalk;
# - modify normal roadway;
# - modify scientific labels.
#
# For each bottom-touching artifact component, inspect nearby
# valid street context. If roadway is dominant, display only
# that existing artifact component as roadway.
# ============================================================

BOTTOM_ARTIFACT_ROAD_ID = LABEL2ID[
    "roadway"
]

BOTTOM_ARTIFACT_SIDEWALK_ID = LABEL2ID[
    "sidewalk"
]

BOTTOM_ARTIFACT_CURB_ID = LABEL2ID[
    "curb_edge"
]

BOTTOM_ARTIFACT_BIKE_ID = LABEL2ID[
    "bike_lane"
]

bottom_artifact_road_lock_mask = np.zeros(
    (H, W),
    dtype=bool
)

bottom_artifact_context_rows = []


n_bottom_art, bottom_lab, bottom_stats, _ = (
    cv2.connectedComponentsWithStats(
        artifact_ignore_mask.astype(np.uint8),
        connectivity=8
    )
)


for cid in range(
    1,
    n_bottom_art
):

    component = (
        bottom_lab == cid
    )

    area = int(
        component.sum()
    )

    if area == 0:
        continue

    x = int(
        bottom_stats[
            cid,
            cv2.CC_STAT_LEFT
        ]
    )

    y = int(
        bottom_stats[
            cid,
            cv2.CC_STAT_TOP
        ]
    )

    w = int(
        bottom_stats[
            cid,
            cv2.CC_STAT_WIDTH
        ]
    )

    h = int(
        bottom_stats[
            cid,
            cv2.CC_STAT_HEIGHT
        ]
    )

    touches_bottom = (
        y + h
        >=
        H - 2
    )

    center_y = (
        y + h / 2
    ) / H

    # Only the true bottom screenshot-smear use case.
    bottom_like = (
        touches_bottom
        and
        center_y >= 0.72
    )

    if not bottom_like:
        continue


    # --------------------------------------------------------
    # Context ring around the EXISTING artifact component.
    # --------------------------------------------------------

    margin_x = max(
        36,
        int(round(W * 0.030))
    )

    margin_y = max(
        20,
        int(round(H * 0.025))
    )

    x0 = max(
        0,
        x - margin_x
    )

    x1 = min(
        W,
        x + w + margin_x
    )

    y0 = max(
        0,
        y - margin_y
    )

    y1 = min(
        H,
        y + h
    )


    local_labels = final_working[
        y0:y1,
        x0:x1
    ]

    local_artifact = artifact_ignore_mask[
        y0:y1,
        x0:x1
    ]

    local_component = component[
        y0:y1,
        x0:x1
    ]


    # Internal bike_lane is counted as roadway evidence because
    # v1.5.5+ merges bike_lane into roadway in final output.
    local_street_valid = (
        ~local_artifact
        &
        ~local_component
        &
        np.isin(
            local_labels,
            [
                BOTTOM_ARTIFACT_ROAD_ID,
                BOTTOM_ARTIFACT_BIKE_ID,
                BOTTOM_ARTIFACT_SIDEWALK_ID,
                BOTTOM_ARTIFACT_CURB_ID,
            ]
        )
    )

    street_labels = local_labels[
        local_street_valid
    ]


    roadway_votes = int(
        np.sum(
            street_labels
            ==
            BOTTOM_ARTIFACT_ROAD_ID
        )
    )

    bike_votes = int(
        np.sum(
            street_labels
            ==
            BOTTOM_ARTIFACT_BIKE_ID
        )
    )

    sidewalk_votes = int(
        np.sum(
            street_labels
            ==
            BOTTOM_ARTIFACT_SIDEWALK_ID
        )
    )

    curb_votes = int(
        np.sum(
            street_labels
            ==
            BOTTOM_ARTIFACT_CURB_ID
        )
    )

    street_votes = int(
        len(
            street_labels
        )
    )

    road_like_votes = (
        roadway_votes
        +
        bike_votes
    )

    road_like_share = (
        road_like_votes / street_votes
        if street_votes > 0
        else 0.0
    )


    roadway_dominant = (
        street_votes >= 24
        and
        road_like_share >= 0.55
        and
        road_like_votes
        >=
        (
            sidewalk_votes
            +
            curb_votes
        )
    )


    if roadway_dominant:

        # DISPLAY ONLY.
        # Scientific final_working remains IGNORE=255 here.
        display_context_labels[
            component
        ] = BOTTOM_ARTIFACT_ROAD_ID

        bottom_artifact_road_lock_mask |= (
            component
        )


    bottom_artifact_context_rows.append({
        "component": cid,
        "pixels": area,
        "center_y": center_y,
        "roadway_votes": roadway_votes,
        "bike_votes_as_road": bike_votes,
        "sidewalk_votes": sidewalk_votes,
        "curb_votes": curb_votes,
        "street_votes": street_votes,
        "road_like_share": road_like_share,
        "roadway_locked": roadway_dominant,
    })


print()
print("========================================")
print("BOTTOM ARTIFACT ROADWAY DISPLAY — v1.5.6")
print("========================================")
print(
    "Roadway-locked existing artifact pixels:",
    int(
        bottom_artifact_road_lock_mask.sum()
    )
)

if bottom_artifact_context_rows:

    display(
        pd.DataFrame(
            bottom_artifact_context_rows
        )
    )


# ------------------------------------------------------------
# v1.5.4A BORDER-FRINGE DISPLAY LOCK
#
# This is the successful v1.5.4 behavior:
# the accepted gray semantic fringe has already passed
# roadway-dominant context validation.
#
# Scientific map remains IGNORE=255.
# Display map alone shows the fringe as roadway.
#
# IMPORTANT:
# Only border_semantic_artifact_mask is modified here.
# No large bottom footprint or sidewalk block is touched.
# ------------------------------------------------------------

if "border_semantic_artifact_mask" in globals():

    display_context_labels[
        border_semantic_artifact_mask
    ] = LABEL2ID[
        "roadway"
    ]


# ------------------------------------------------------------
# v1.5.6 BOTTOM ARTIFACT DISPLAY QA
# ------------------------------------------------------------

assert not np.any(
    bottom_artifact_road_lock_mask
    &
    (
        display_context_labels
        !=
        BOTTOM_ARTIFACT_ROAD_ID
    )
), (
    "BOTTOM ARTIFACT ROAD LOCK FAILURE: a roadway-dominant "
    "confirmed Google artifact is not displayed as roadway."
)

# Scientific data is still IGNORE for those same pixels.
assert np.all(
    final_working[
        bottom_artifact_road_lock_mask
    ]
    ==
    IGNORE_ID
), (
    "BOTTOM ARTIFACT SCIENTIFIC FAILURE: display-only roadway "
    "lock leaked into scientific labels."
)


# ------------------------------------------------------------
# 3. HARD QA ASSERTIONS
# ------------------------------------------------------------

assert np.all(
    final_working[artifact_ignore_mask] == IGNORE_ID
), "STRICT IGNORE FAILURE: artifact pixels received semantic labels."

if "border_semantic_artifact_mask" in globals():

    assert np.all(
        final_working[
            border_semantic_artifact_mask
        ]
        ==
        IGNORE_ID
    ), (
        "BORDER FRINGE SCIENTIFIC FAILURE: "
        "captured gray fringe is not IGNORE=255."
    )

    assert not np.any(
        border_semantic_artifact_mask
        &
        (
            display_context_labels
            !=
            LABEL2ID[
                "roadway"
            ]
        )
    ), (
        "BORDER FRINGE DISPLAY FAILURE: "
        "captured gray fringe is not displayed as roadway."
    )

if "pole_fixture" in LABEL2ID:
    assert not np.any(
        artifact_ignore_mask
        &
        (final_working == LABEL2ID["pole_fixture"])
    ), "STRICT IGNORE FAILURE: Google artifact became pole_fixture."

if "traffic_sign_signal" in LABEL2ID:
    assert not np.any(
        artifact_ignore_mask
        &
        (final_working == LABEL2ID["traffic_sign_signal"])
    ), "STRICT IGNORE FAILURE: Google artifact became traffic_sign_signal."

print("========================================")
print("STREET VIEW ARTIFACT FINAL LOCK")
print("========================================")
print("Detected artifact pixels:", artifact_detected_pixels)
print("Scientific semantic assignments inside artifact:", 0)
print("Permanent IGNORE pixels:", int(artifact_ignore_mask.sum()))

if "border_semantic_artifact_mask" in globals():

    print(
        "v1.5.4 border-fringe pixels:",
        int(
            border_semantic_artifact_mask.sum()
        )
    )

    print(
        "Border-fringe non-road display pixels:",
        int(
            np.sum(
                border_semantic_artifact_mask
                &
                (
                    display_context_labels
                    !=
                    LABEL2ID[
                        "roadway"
                    ]
                )
            )
        )
    )
print(
    "Bottom artifact road-lock non-road display pixels:",
    int(
        np.sum(
            bottom_artifact_road_lock_mask
            &
            (
                display_context_labels
                !=
                BOTTOM_ARTIFACT_ROAD_ID
            )
        )
    )
)

print("✓ Artifact cannot contribute to pole_fixture")
print("✓ Artifact cannot contribute to traffic_sign_signal")
print("✓ Artifact is excluded from all class percentages")
print("✓ Row-aware roadway/sidewalk contextual fill exists ONLY for visualization")


In [ ]:
# ============================================================
# COMMIT + STRICT IGNORE VISUAL CHECK — v1.5
# ============================================================

final_v010 = final_working.copy()

# Re-assert permanent scientific IGNORE at the commit boundary.
if "artifact_ignore_mask" in globals():
    final_v010[artifact_ignore_mask] = IGNORE_ID

rgb_v010_scientific = render_taxonomy(final_v010)

# Scientific IGNORE shown dark gray in this QA panel only.
rgb_v010_scientific[
    final_v010 == IGNORE_ID
] = (35, 35, 35)

original_np = np.array(image)

# Display-clean semantic map is separate from scientific labels.
display_label_map = (
    display_context_labels.copy()
    if "display_context_labels" in globals()
    else final_v010.copy()
)

display_label_map[
    display_label_map == IGNORE_ID
] = UNKNOWN_ID

rgb_v010 = render_taxonomy(display_label_map)

overlay_v010 = original_np.copy()

normal_classified = (
    (final_v010 != UNKNOWN_ID)
    &
    (final_v010 != IGNORE_ID)
)

overlay_v010[normal_classified] = (
    original_np[normal_classified] * 0.42
    +
    rgb_v010[normal_classified] * 0.58
).astype(np.uint8)

# Artifact area: use display-only contextual semantic color,
# not the original screenshot pixels.
if "artifact_ignore_mask" in globals():
    overlay_v010[artifact_ignore_mask] = rgb_v010[artifact_ignore_mask]

# Hard post-commit assertion.
if "artifact_ignore_mask" in globals():
    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)

plt.figure(figsize=(22, 8))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title(f"{TEST_ID} — Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rgb_v010_scientific)
plt.title(f"{TEST_ID} — Scientific Labels (dark gray = IGNORE)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(rgb_v010)
plt.title(f"{TEST_ID} — Display Clean (context fill only)")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Scientific map keeps Google/UI residual as IGNORE=255")
print("✓ Display map fills the hole without changing scientific statistics")


In [ ]:
# ============================================================
# ROADWAY PRECEDENCE CLEANUP — v1.5.7
#
# Target:
# sidewalk-colored fragments located INSIDE the roadway.
#
# Key principle:
# Roadway may reclaim a sidewalk component only when multiple
# independent spatial / semantic checks agree.
#
# This is component-level, not a global sidewalk rewrite.
# ============================================================

ROAD_CLEAN_ID = LABEL2ID["roadway"]
SIDEWALK_CLEAN_ID = LABEL2ID["sidewalk"]

current_sidewalk_v157 = (
    final_working == SIDEWALK_CLEAN_ID
)

# ------------------------------------------------------------
# Road / sidewalk evidence
# ------------------------------------------------------------

road_semantic_support_v157 = (
    MAP_ROAD
    |
    ADE_ROAD_V2
)

strong_sidewalk_support_v157 = (
    MAP_SIDEWALK
    &
    ADE_SIDEWALK_V2
)

# Keep genuine sidewalk near building / interface.
building_interface_near_v157 = cv2.dilate(
    building_context.astype(np.uint8),
    cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (35, 25)
    ),
    iterations=1
) > 0

# Street foreground zone.
lower_foreground_v157 = np.zeros(
    (H, W),
    dtype=bool
)

lower_foreground_v157[
    int(H * 0.50):H,
    :
] = True

# Candidate sidewalk that might actually be roadway.
sidewalk_road_candidate_v157 = (
    current_sidewalk_v157
    &
    lower_foreground_v157
    &
    road_semantic_support_v157
)

# Do not touch strong true-sidewalk evidence.
sidewalk_road_candidate_v157 &= ~strong_sidewalk_support_v157

# Do not touch building-edge sidewalk.
sidewalk_road_candidate_v157 &= ~building_interface_near_v157

# Never touch artifact scientific pixels.
if "artifact_ignore_mask" in globals():
    sidewalk_road_candidate_v157 &= ~artifact_ignore_mask


# ------------------------------------------------------------
# Connected-component validation
# ------------------------------------------------------------

n_sw, sw_lab, sw_stats, _ = cv2.connectedComponentsWithStats(
    sidewalk_road_candidate_v157.astype(np.uint8),
    connectivity=8
)

false_sidewalk_to_road_v157 = np.zeros(
    (H, W),
    dtype=bool
)

road_cleanup_rows_v157 = []

for cid in range(1, n_sw):

    comp = (
        sw_lab == cid
    )

    area = int(
        comp.sum()
    )

    if area == 0:
        continue

    x = int(
        sw_stats[
            cid,
            cv2.CC_STAT_LEFT
        ]
    )

    y = int(
        sw_stats[
            cid,
            cv2.CC_STAT_TOP
        ]
    )

    w = int(
        sw_stats[
            cid,
            cv2.CC_STAT_WIDTH
        ]
    )

    h = int(
        sw_stats[
            cid,
            cv2.CC_STAT_HEIGHT
        ]
    )

    area_ratio = (
        area / (H * W)
    )

    width_ratio = (
        w / W
    )

    height_ratio = (
        h / H
    )

    center_y = (
        y + h / 2
    ) / H

    # --------------------------------------------------------
    # Local ring around the component.
    # --------------------------------------------------------

    margin_x = max(
        24,
        int(round(W * 0.022))
    )

    margin_y = max(
        18,
        int(round(H * 0.022))
    )

    x0 = max(
        0,
        x - margin_x
    )

    x1 = min(
        W,
        x + w + margin_x
    )

    y0 = max(
        0,
        y - margin_y
    )

    y1 = min(
        H,
        y + h + margin_y
    )

    local_labels = final_working[
        y0:y1,
        x0:x1
    ]

    local_comp = comp[
        y0:y1,
        x0:x1
    ]

    valid_local = (
        ~local_comp
        &
        (
            local_labels != IGNORE_ID
        )
    )

    local_valid_labels = local_labels[
        valid_local
    ]

    road_votes = int(
        np.sum(
            local_valid_labels
            ==
            ROAD_CLEAN_ID
        )
    )

    sidewalk_votes = int(
        np.sum(
            local_valid_labels
            ==
            SIDEWALK_CLEAN_ID
        )
    )

    bike_votes = int(
        np.sum(
            local_valid_labels
            ==
            LABEL2ID["bike_lane"]
        )
    )

    road_like_votes = (
        road_votes
        +
        bike_votes
    )

    street_votes = (
        road_like_votes
        +
        sidewalk_votes
    )

    road_like_share = (
        road_like_votes / street_votes
        if street_votes > 0
        else 0.0
    )

    road_support_ratio = float(
        road_semantic_support_v157[
            comp
        ].mean()
    )

    map_sidewalk_ratio = float(
        MAP_SIDEWALK[
            comp
        ].mean()
    )

    ade_sidewalk_ratio = float(
        ADE_SIDEWALK_V2[
            comp
        ].mean()
    )

    # --------------------------------------------------------
    # Two accepted false-sidewalk patterns:
    #
    # A. Small isolated road fragment
    # B. Thin / elongated road-edge strip
    # --------------------------------------------------------

    small_isolated = (
        area_ratio <= 0.012
        and
        road_like_share >= 0.62
        and
        road_support_ratio >= 0.45
    )

    thin_road_strip = (
        height_ratio <= 0.10
        and
        (
            width_ratio >= 0.08
            or
            w >= 3 * max(h, 1)
        )
        and
        road_like_share >= 0.58
        and
        road_support_ratio >= 0.45
    )

    weak_true_sidewalk = (
        map_sidewalk_ratio < 0.40
        or
        ade_sidewalk_ratio < 0.40
    )

    foreground_position_ok = (
        center_y >= 0.55
    )

    accept_as_road = (
        foreground_position_ok
        and
        weak_true_sidewalk
        and
        (
            small_isolated
            or
            thin_road_strip
        )
    )

    if accept_as_road:
        false_sidewalk_to_road_v157[
            comp
        ] = True

    road_cleanup_rows_v157.append({
        "component": cid,
        "pixels": area,
        "width": w,
        "height": h,
        "area_ratio": area_ratio,
        "width_ratio": width_ratio,
        "height_ratio": height_ratio,
        "center_y": center_y,
        "road_like_share": road_like_share,
        "road_semantic_support": road_support_ratio,
        "map_sidewalk_support": map_sidewalk_ratio,
        "ade_sidewalk_support": ade_sidewalk_ratio,
        "small_isolated": small_isolated,
        "thin_road_strip": thin_road_strip,
        "accepted_as_roadway": accept_as_road,
    })


# ------------------------------------------------------------
# Install correction
# ------------------------------------------------------------

sidewalk_before_v157 = int(
    np.sum(
        final_working
        ==
        SIDEWALK_CLEAN_ID
    )
)

road_before_v157 = int(
    np.sum(
        final_working
        ==
        ROAD_CLEAN_ID
    )
)

final_working[
    false_sidewalk_to_road_v157
] = ROAD_CLEAN_ID

# Keep display map synchronized if already created.
if "display_context_labels" in globals():

    display_context_labels[
        false_sidewalk_to_road_v157
    ] = ROAD_CLEAN_ID


sidewalk_after_v157 = int(
    np.sum(
        final_working
        ==
        SIDEWALK_CLEAN_ID
    )
)

road_after_v157 = int(
    np.sum(
        final_working
        ==
        ROAD_CLEAN_ID
    )
)


print("========================================")
print("ROADWAY PRECEDENCE CLEANUP — v1.5.7")
print("========================================")
print(
    "False sidewalk pixels → roadway:",
    int(
        false_sidewalk_to_road_v157.sum()
    )
)
print(
    "Sidewalk pixels:",
    sidewalk_before_v157,
    "→",
    sidewalk_after_v157
)
print(
    "Roadway pixels:",
    road_before_v157,
    "→",
    road_after_v157
)

if road_cleanup_rows_v157:
    display(
        pd.DataFrame(
            road_cleanup_rows_v157
        )
    )


# ------------------------------------------------------------
# QA:
# corrected pixels must no longer remain sidewalk.
# ------------------------------------------------------------

assert not np.any(
    false_sidewalk_to_road_v157
    &
    (
        final_working
        ==
        SIDEWALK_CLEAN_ID
    )
), (
    "ROADWAY PRECEDENCE FAILURE: accepted false sidewalk "
    "component still remains sidewalk."
)

print(
    "✓ v1.5.7 roadway precedence cleanup applied"
)


# ============================================================
# SHARE-READY AUTHORITATIVE STATE COMMIT — v1.5.7.1
#
# v1.5.7 performs roadway cleanup after the earlier final_v010
# snapshot.  Re-commit here so scientific QA / stats / exports
# cannot drift from the final visualization state.
# ============================================================

final_v010 = final_working.copy()

if "artifact_ignore_mask" in globals():
    final_v010[
        artifact_ignore_mask
    ] = IGNORE_ID

# Keep display state aligned with the corrected semantic state
# while preserving display-only artifact context fill.
if "display_context_labels" in globals():
    non_artifact_pixels_v1571 = np.ones(
        (H, W),
        dtype=bool
    )

    if "artifact_ignore_mask" in globals():
        non_artifact_pixels_v1571 &= ~artifact_ignore_mask

    display_context_labels[
        non_artifact_pixels_v1571
    ] = final_working[
        non_artifact_pixels_v1571
    ]

print(
    "✓ Authoritative scientific state re-committed after roadway cleanup"
)


In [ ]:
# ============================================================
# FINAL TAXONOMY MERGE — v1.5.5
# bike_lane → roadway
#
# IMPORTANT:
# The internal bike_lane ID remains declared so earlier model /
# fusion code does not break.
#
# But from THIS POINT FORWARD:
# - scientific final labels
# - class percentages
# - exported label PNG
# - Combined Semantic Mask
# - final legend
#
# contain NO independent bike_lane class.
# ============================================================

ROADWAY_ID = LABEL2ID["roadway"]
BIKE_LANE_ID = LABEL2ID["bike_lane"]

bike_lane_pixels_before_merge = int(
    np.sum(
        final_working == BIKE_LANE_ID
    )
)

final_working[
    final_working == BIKE_LANE_ID
] = ROADWAY_ID

# Keep compatibility states aligned if they exist.
if "final_v010" in globals():
    final_v010[
        final_v010 == BIKE_LANE_ID
    ] = ROADWAY_ID

if "display_context_labels" in globals():
    display_context_labels[
        display_context_labels == BIKE_LANE_ID
    ] = ROADWAY_ID

print("========================================")
print("FINAL TAXONOMY MERGE — v1.5.5")
print("========================================")
print("bike_lane pixels merged into roadway:", bike_lane_pixels_before_merge)
print(
    "bike_lane pixels remaining in final_working:",
    int(np.sum(final_working == BIKE_LANE_ID))
)

assert not np.any(
    final_working == BIKE_LANE_ID
), (
    "BIKE-LANE MERGE FAILURE: bike_lane is still present "
    "in the final scientific label map."
)

print("✓ bike_lane is no longer an independent final output class")

# v1.5.7.1: final_v010 is the authoritative scientific export
# state and must equal final_working everywhere except that
# artifact pixels are explicitly IGNORE=255 in both.
assert np.array_equal(
    final_v010,
    final_working
), (
    "FINAL STATE DRIFT: final_v010 and final_working differ "
    "after final taxonomy merge."
)

print(
    "✓ final_v010 == final_working at final taxonomy boundary"
)


In [ ]:
# ============================================================
# CURRENT QA — v1.5.7.1 SHARE-READY AUDITED
# ============================================================

valid = final_v010 != IGNORE_ID
valid_count = max(int(valid.sum()), 1)

def current_pct(class_name):
    cid = LABEL2ID[class_name]
    return np.sum(valid & (final_v010 == cid)) / valid_count * 100

qa_classes = [
    "roadway",
    "sidewalk",
    "stoop_stair",
    "awning_canopy",
    "wall_ledge",
    "sidewalk_shed_scaffold",
    "fence_railing",
    "traffic_cone_barrel",
    "pole_fixture",
    "traffic_sign_signal",
    "tree",
    "upper_building_facade",
    "upper_building_glazing",
    "ground_floor_glazing",
    "other_unknown",
]

qa_df = pd.DataFrame([
    {"class_name": name, "percentage": current_pct(name)}
    for name in qa_classes
])
display(qa_df)

# -----------------------------
# Scientific / schema invariants
# -----------------------------
declared_ids = set(CLASS_NAMES.keys())
observed_ids = set(np.unique(final_v010[valid]).astype(int).tolist())

assert observed_ids.issubset(declared_ids), (
    f"Undeclared label IDs found: {sorted(observed_ids - declared_ids)}"
)
assert NUM_CLASSES == TAXONOMY["num_classes"] == len(CVAT_LABELS) == 30
assert LABEL2ID["traffic_cone_barrel"] == 29
assert "tree_canopy" not in LABEL2ID and "tree_trunk" not in LABEL2ID

if "artifact_ignore_mask" in globals():
    assert np.all(final_v010[artifact_ignore_mask] == IGNORE_ID)

total_pct = sum(
    np.sum(valid & (final_v010 == cid)) / valid_count * 100
    for cid in CLASS_NAMES
)

print("Tree:", current_pct("tree"), "%")
print("Sidewalk:", current_pct("sidewalk"), "%")
print("Stoop / stair:", current_pct("stoop_stair"), "%")
print("Awning / canopy:", current_pct("awning_canopy"), "%")
print("Wall / ledge:", current_pct("wall_ledge"), "%")
print("Upper façade:", current_pct("upper_building_facade"), "%")
print("Upper glazing:", current_pct("upper_building_glazing"), "%")
print("Scaffold:", current_pct("sidewalk_shed_scaffold"), "%")
print("Fence / railing:", current_pct("fence_railing"), "%")
print("Ground glazing:", current_pct("ground_floor_glazing"), "%")
print("Other unknown:", current_pct("other_unknown"), "%")
print("Declared-class percentage sum:", total_pct, "%")

assert abs(total_pct - 100.0) < 1e-6
print("✓ SHARE QA PASSED: schema, labels, IGNORE policy, and percentages are internally consistent")


# v1.5.5 final-class merge QA
assert current_pct("bike_lane") == 0.0, (
    "bike_lane must be merged into roadway before final statistics."
)

print(
    "Bike lane exposed in final output:",
    "NO"
)


# Share-ready state consistency
assert np.array_equal(
    final_v010,
    final_working
), "SHARE QA FAILURE: final_v010 != final_working"

print("✓ SHARE STATE QA PASSED: scientific/export state matches final inference state")


In [ ]:
# ============================================================
# STRICT-IGNORE DISPLAY — v1.5
#
# SCIENTIFIC:
#   final_v010 keeps Google/UI artifact as IGNORE=255.
#
# DISPLAY:
#   display_label_map uses contextual broad-surface fill only.
#   It is never used for statistics.
# ============================================================

display_label_map = (
    display_context_labels.copy()
    if "display_context_labels" in globals()
    else final_v010.copy()
)

display_label_map[
    display_label_map == IGNORE_ID
] = UNKNOWN_ID

rgb_display_clean = render_taxonomy(display_label_map)

original_np = np.array(image)
overlay_display_clean = original_np.copy()

normal_classified = (
    (final_v010 != UNKNOWN_ID)
    &
    (final_v010 != IGNORE_ID)
)

overlay_display_clean[normal_classified] = (
    original_np[normal_classified] * 0.42
    +
    rgb_display_clean[normal_classified] * 0.58
).astype(np.uint8)

if "artifact_ignore_mask" in globals():
    overlay_display_clean[artifact_ignore_mask] = (
        rgb_display_clean[artifact_ignore_mask]
    )

plt.figure(figsize=(22, 8))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title(f"{TEST_ID} — Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rgb_display_clean)
plt.title(f"{TEST_ID} — Combined Semantic Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay_display_clean)
plt.title(f"{TEST_ID} — Strict-Ignore Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

print("✓ Google/UI pixels are excluded from scientific labels")
print("✓ Display fill cannot create pole/sign/person/vehicle/cone artifacts")


In [ ]:
# ============================================================
# USE STRICT-IGNORE DISPLAY VERSION FOR VISUAL EXPORT
# ============================================================

rgb_v010 = rgb_display_clean.copy()
overlay_v010 = overlay_display_clean.copy()

print("✓ RGB / Overlay export uses display-only context fill")
print("✓ Scientific label export still contains IGNORE=255")


In [ ]:
# ============================================================
# FINAL STABLE EXPORT — v1.5.7.1 SHARE-READY AUDITED
#
# Scientific label map:
# - Google/UI screenshot residuals remain IGNORE=255
# - IGNORE pixels are excluded from every percentage denominator
# - artifact pixels NEVER count as pole/road/sidewalk/etc.
#
# Visual RGB / overlay:
# - ignored artifact holes are context-filled for readability only
# - visual fill does NOT affect statistics
# ============================================================

from pathlib import Path
from PIL import Image
import json
import numpy as np
import pandas as pd

STABLE_SHARE_VERSION = "vision_model_segmentation_taxonomy_v1.5.7.1_share_ready_audited"

TEST_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "v1_5_7_1_share_ready_audited"
    / TEST_ID
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Statistics: IGNORE excluded from denominator.
# ------------------------------------------------------------

valid_mask = final_v010 != IGNORE_ID
valid_pixel_count = max(int(valid_mask.sum()), 1)

rows_final = []

for class_id in sorted(CLASS_NAMES.keys()):
    class_name = CLASS_NAMES[class_id]

    # bike_lane is an internal compatibility ID only.
    # Final share outputs expose it as roadway.
    if class_name == "bike_lane":
        continue

    pixel_count = int(np.sum(valid_mask & (final_v010 == class_id)))
    percentage = pixel_count / valid_pixel_count * 100.0

    rows_final.append({
        "class_id": class_id,
        "class_name": class_name,
        "pixels": pixel_count,
        "percentage": percentage
    })

stats_final = pd.DataFrame(rows_final).sort_values(
    ["percentage", "class_id"],
    ascending=[False, True]
).reset_index(drop=True)

unknown_pct = float(
    np.sum(valid_mask & (final_v010 == LABEL2ID["other_unknown"]))
    / valid_pixel_count
    * 100.0
)

ignore_pct_of_full_image = float(
    np.mean(final_v010 == IGNORE_ID) * 100.0
)

coverage_pct = 100.0 - unknown_pct

artifact_detected_pct = float(
    artifact_ignore_mask.mean() * 100.0
) if "artifact_ignore_mask" in globals() else 0.0

artifact_ignored_pixels = int(
    artifact_ignore_mask.sum()
) if "artifact_ignore_mask" in globals() else 0

# ------------------------------------------------------------
# HARD SCIENTIFIC QA
# ------------------------------------------------------------

if "artifact_ignore_mask" in globals():
    assert np.all(
        final_v010[artifact_ignore_mask] == IGNORE_ID
    )

artifact_pole_pixels = 0
if "artifact_ignore_mask" in globals() and "pole_fixture" in LABEL2ID:
    artifact_pole_pixels = int(np.sum(
        artifact_ignore_mask
        &
        (final_v010 == LABEL2ID["pole_fixture"])
    ))

assert artifact_pole_pixels == 0, (
    "Google/UI artifact contamination detected in pole_fixture."
)

# Full label-schema audit at export boundary.
declared_ids = set(CLASS_NAMES.keys())
observed_valid_ids = set(np.unique(final_v010[valid_mask]).astype(int).tolist())
assert observed_valid_ids.issubset(declared_ids), (
    f"Undeclared final labels: {sorted(observed_valid_ids - declared_ids)}"
)
assert NUM_CLASSES == TAXONOMY["num_classes"] == len(CVAT_LABELS) == 30

# ------------------------------------------------------------
# SHARE-READY FINAL STATE QA
# ------------------------------------------------------------

assert np.array_equal(
    final_v010,
    final_working
), (
    "EXPORT BLOCKED: scientific final_v010 differs from "
    "final_working."
)

assert not np.any(
    final_v010 == LABEL2ID["bike_lane"]
), (
    "EXPORT BLOCKED: bike_lane pixels remain in final output."
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

image.save(TEST_OUTPUT_DIR / f"{TEST_ID}_ORIGINAL.png")

# Scientific labels: 255 remains 255.
Image.fromarray(final_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_LABEL_MAP_STABLE.png"
)

Image.fromarray(final_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_PSEUDO_LABEL_STABLE.png"
)

# Display-clean outputs.
Image.fromarray(rgb_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_RGB_CLEAN.png"
)

Image.fromarray(overlay_v010.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_OVERLAY_CLEAN.png"
)

# Optional display-only label map for auditability.
Image.fromarray(display_label_map.astype(np.uint8)).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_DISPLAY_CONTEXT_LABEL_MAP.png"
)

artifact_mask_export = (
    artifact_ignore_mask.astype(np.uint8) * 255
    if "artifact_ignore_mask" in globals()
    else np.zeros((H, W), dtype=np.uint8)
)

Image.fromarray(artifact_mask_export).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_STREETVIEW_ARTIFACT_IGNORE_MASK.png"
)

Image.fromarray(
    (final_v010 == IGNORE_ID).astype(np.uint8) * 255
).save(
    TEST_OUTPUT_DIR / f"{TEST_ID}_IGNORE_MASK.png"
)

stats_final.to_csv(
    TEST_OUTPUT_DIR / f"{TEST_ID}_STATISTICS_STABLE.csv",
    index=False,
    encoding="utf-8-sig"
)

diagnostics_final = {
    "test_id": TEST_ID,
    "stable_share_version": STABLE_SHARE_VERSION,
    "coverage_pct_excluding_ignore": coverage_pct,
    "unknown_pct_excluding_ignore": unknown_pct,
    "ignore_pct_of_full_image": ignore_pct_of_full_image,
    "streetview_artifact_ignored_pct": artifact_detected_pct,
    "streetview_artifact_ignored_pixels": artifact_ignored_pixels,
    "streetview_artifact_pixels_classified_as_pole": artifact_pole_pixels,
    "valid_pixels": valid_pixel_count,
    "full_image_pixels": int(final_v010.size),
    "declared_num_classes": NUM_CLASSES,
    "observed_valid_label_ids": sorted(observed_valid_ids),
    "scientific_rule": (
        "Google Street View screenshot/UI residual pixels are IGNORE=255 "
        "and excluded from taxonomy percentages."
    )
}

with open(
    TEST_OUTPUT_DIR / f"{TEST_ID}_DIAGNOSTICS_STABLE.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        diagnostics_final,
        f,
        ensure_ascii=False,
        indent=2
    )

manifest_final = {
    "stable_share_version": STABLE_SHARE_VERSION,
    "test_id": TEST_ID,
    "taxonomy_version": TAXONOMY.get("taxonomy_version", "unknown"),
    "ignore_index": IGNORE_ID,
    "declared_num_classes": NUM_CLASSES,
    "traffic_cone_barrel_id": LABEL2ID["traffic_cone_barrel"],
    "bike_lane_policy": "merged_into_roadway_final_output",
    "architecture": "v0.9_semantic_backbone_plus_permissioned_refinements",
    "tree_refinement_policy": "backbone_locked_no_broad_dino_sam_overwrite",
    "stoop_policy": "entrance_anchored_precision",
    "google_artifact_policy": "strict_ignore",
    "google_artifact_display_policy": "context_fill_visualization_only",
}

with open(
    TEST_OUTPUT_DIR / f"{TEST_ID}_RUN_MANIFEST_STABLE.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest_final,
        f,
        ensure_ascii=False,
        indent=2
    )

print("========================================")
print("FINAL STABLE EXPORT — v1.5.7.1 SHARE-READY AUDITED")
print("========================================")
print("Output:", TEST_OUTPUT_DIR)
print("Valid pixels:", valid_pixel_count)
print("IGNORE % of full image:", ignore_pct_of_full_image)
print("Google/UI ignored pixels:", artifact_ignored_pixels)
print("Google/UI pixels classified as pole:", artifact_pole_pixels)
print("✓ STRICT IGNORE + TAXONOMY SCHEMA QA PASSED")


In [ ]:
# ============================================================
# FINAL SUMMARY FIGURE — v1.5.7 LAYOUT-SAFE
#
# Requirements:
# - bike_lane is merged into roadway
# - image panels never overlap legend
# - legend items never overlap each other
# - figure height adapts to number of visible classes
# ============================================================

import math
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# ------------------------------------------------------------
# 1. Guarantee final merge before rendering
# ------------------------------------------------------------

ROADWAY_ID = LABEL2ID["roadway"]
BIKE_LANE_ID = LABEL2ID["bike_lane"]

final_working[
    final_working == BIKE_LANE_ID
] = ROADWAY_ID

if "display_context_labels" not in globals():
    display_context_labels = final_working.copy()

display_context_labels[
    display_context_labels == BIKE_LANE_ID
] = ROADWAY_ID

# Scientific percentages exclude IGNORE.
valid_mask_v155 = (
    final_working != IGNORE_ID
)

valid_total_v155 = int(
    valid_mask_v155.sum()
)

# ------------------------------------------------------------
# 2. Compute visible final classes
# ------------------------------------------------------------

final_stats_v155 = []

for class_id, class_name in CLASS_NAMES.items():

    # bike_lane is intentionally hidden / merged.
    if class_name == "bike_lane":
        continue

    px = int(
        np.sum(
            (final_working == class_id)
            &
            valid_mask_v155
        )
    )

    if px <= 0:
        continue

    pct = (
        100.0 * px / valid_total_v155
        if valid_total_v155 > 0
        else 0.0
    )

    final_stats_v155.append(
        (
            class_id,
            class_name,
            px,
            pct
        )
    )

# Descending percentage, same research-friendly ordering.
final_stats_v155.sort(
    key=lambda x: x[3],
    reverse=True
)

# ------------------------------------------------------------
# 3. Render semantic display
# ------------------------------------------------------------

semantic_rgb_v155 = render_taxonomy(
    display_context_labels
)

original_rgb_v155 = np.array(
    image
)

# ------------------------------------------------------------
# 4. Adaptive layout
#
# Top row = two images
# Bottom region = legend
#
# The legend region has its OWN GridSpec row, so overlap with
# image panels is geometrically impossible.
# ------------------------------------------------------------

n_items = len(
    final_stats_v155
)

# Two columns normally; three if many classes.
legend_cols = (
    3
    if n_items >= 21
    else 2
)

legend_rows = max(
    1,
    math.ceil(
        n_items / legend_cols
    )
)

# Height grows with legend rows.
figure_height = (
    7.4
    +
    legend_rows * 0.56
)

fig = plt.figure(
    figsize=(
        17.5,
        figure_height
    ),
    constrained_layout=False
)

gs = gridspec.GridSpec(
    nrows=2,
    ncols=2,
    figure=fig,
    height_ratios=[
        1.0,
        max(
            0.32,
            legend_rows * 0.095
        )
    ],
    hspace=0.16,
    wspace=0.02
)

ax_original = fig.add_subplot(
    gs[0, 0]
)

ax_mask = fig.add_subplot(
    gs[0, 1]
)

ax_legend = fig.add_subplot(
    gs[1, :]
)

# ------------------------------------------------------------
# 5. Top image panels
# ------------------------------------------------------------

ax_original.imshow(
    original_rgb_v155
)

ax_original.set_title(
    "Original SVI",
    fontsize=28,
    pad=14
)

ax_original.axis(
    "off"
)


ax_mask.imshow(
    semantic_rgb_v155
)

ax_mask.set_title(
    "Combined Semantic Mask",
    fontsize=28,
    pad=14
)

ax_mask.axis(
    "off"
)

# ------------------------------------------------------------
# 6. Legend panel — isolated from images
# ------------------------------------------------------------

ax_legend.axis(
    "off"
)

legend_handles = []
legend_labels = []

for class_id, class_name, px, pct in final_stats_v155:

    color = np.array(
        PALETTE[
            class_id
        ],
        dtype=float
    ) / 255.0

    legend_handles.append(
        Patch(
            facecolor=color,
            edgecolor="none"
        )
    )

    display_name = class_name.replace(
        "_",
        " "
    )

    legend_labels.append(
        f"{display_name} — {pct:.2f}%"
    )


legend = ax_legend.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        1.0
    ),
    ncol=legend_cols,
    frameon=False,
    fontsize=15.5,
    handlelength=1.25,
    handleheight=0.9,
    columnspacing=2.4,
    labelspacing=0.92,
    borderaxespad=0.0
)

# ------------------------------------------------------------
# 7. Hard layout QA
# ------------------------------------------------------------

fig.canvas.draw()

renderer = fig.canvas.get_renderer()

image_bottom_y = min(
    ax_original.get_position().y0,
    ax_mask.get_position().y0
)

legend_top_y = ax_legend.get_position().y1

# GridSpec should guarantee a positive vertical gap.
assert legend_top_y < image_bottom_y, (
    "LAYOUT FAILURE: legend region overlaps image panels."
)

# bike_lane must not be present in final legend.
assert all(
    "bike lane" not in label.lower()
    for label in legend_labels
), (
    "FINAL LEGEND FAILURE: bike_lane is still exposed."
)

assert not np.any(
    final_working == BIKE_LANE_ID
), (
    "FINAL LABEL FAILURE: bike_lane pixels remain after merge."
)

print(
    "Final visible classes:",
    n_items
)

print(
    "Legend columns:",
    legend_cols
)

print(
    "Legend rows:",
    legend_rows
)

print(
    "Figure height:",
    round(
        figure_height,
        2
    )
)

print(
    "bike_lane visible in legend:",
    False
)

print(
    "bike_lane pixels remaining:",
    int(
        np.sum(
            final_working
            ==
            BIKE_LANE_ID
        )
    )
)

plt.show()


## v1.5 share boundary

### Architecture

`v0.9 semantic backbone → permissioned local refinements`

### v1.5 invariants

- 30 formal taxonomy classes; IDs unchanged.
- Google Street View UI residuals remain scientific `IGNORE=255`.
- Combined-mask artifact visualization cannot use `other_unknown` as a donor.
- Bottom artifact visualization first restores roadway / sidewalk / bike-lane / curb from the same image row.
- Solid residential/community boundary walls are represented by `wall_ledge`.
- Openwork metal fences remain `fence_railing`.
- Entrance awning recovery is geometry-gated and cannot overwrite roadway/sidewalk.
- Broad foliage DINO/SAM masks still have zero write permission.
- Only strict woody trunk/branch structure may be recovered into `tree`.
- Precise woody branches may cross protected façade/glazing.
- Any tree-on-glazing pixel not approved by the woody gate is restored to the v0.9 semantic backbone.
- Pole recovery cannot steal approved woody tree pixels.
- No hard-coded tree count is used; the pipeline must generalize across images.
- No generic `other_unknown` filling is included.

**Candidate version:** validate v1.5 on the previous successful scenes plus several new street scenes before replacing the GitHub stable baseline.


### v1.5.4A — border-fringe-only transplant

Base remains the uploaded `v1.5 LOCAL_RECOVERY_BALANCED`.

Imported from the successful v1.5.4:
- `BORDER SEMANTIC ANOMALY / FRINGE DETECTOR`
- border-fringe scientific `IGNORE=255`
- border-fringe display lock to `roadway`

Explicitly excluded:
- `FULL BOTTOM-FOOTPRINT COMPLETION`
- whole-bottom roadway lock
- large artifact footprint expansion

This is intended to keep the successful removal of the gray lower-border
`upper_building_facade` residual without importing the later logic that produced
the unwanted purple sidewalk block.


### v1.5.5 final taxonomy / layout policy

- `bike_lane` remains an internal compatibility ID only.
- Before final QA/export/rendering, every `bike_lane` pixel is remapped to `roadway`.
- Final percentages therefore count former bike-lane pixels under `roadway`.
- `bike_lane` is omitted from the final legend.
- Final summary uses a dedicated GridSpec row for legend content.
- Legend height and column count adapt to the number of visible classes.
- Legend content cannot overlap the Original SVI or Combined Semantic Mask panels.


### v1.5.6 local precedence guards

**Bottom Google artifact display**
- operates only on pixels already inside `artifact_ignore_mask`;
- no footprint expansion;
- if a bottom-touching component has roadway-dominant context, that artifact
  component is displayed as `roadway`;
- scientific label remains `IGNORE=255`.

**Scaffold precedence**
- `accepted_scaffold` plus nearby `strong_scaffold_dino` forms a compact
  evidence-backed `scaffold_protected_zone`;
- `awning_canopy` and `wall_ledge` recovery are prohibited inside this zone;
- any such conflict is restored to `sidewalk_shed_scaffold`.


### v1.5.7 roadway precedence cleanup

Only `sidewalk` components are reconsidered.

A component is changed to `roadway` only when:
- it lies in the lower street foreground;
- Mapillary/ADE provide roadway evidence;
- strong sidewalk agreement is absent;
- it is away from the building/interface edge;
- its local neighborhood is roadway-dominant;
- its geometry matches either a small isolated fragment or a thin roadway strip.

This targets false pink sidewalk fragments inside the road without globally
removing real sidewalk.


### v1.5.7.1 share-ready audit

- Internal runtime schema remains 30 IDs for backward compatibility.
- User-facing final output exposes `bike_lane` only through `roadway`.
- `final_working` and `final_v010` are synchronized after the v1.5.7 roadway cleanup.
- QA, CSV statistics, scientific label PNG, diagnostics, and final figure use the same final semantic state.
- No API keys, passwords, access tokens, or user-specific Drive paths are embedded.
- Notebook expects Google Colab with a GPU and accepts one image per run.
